# 📗 스키마 설계: 무엇을 노드로 두고 무엇을 관계로 이을까요?

지난 교안에서는 **넣는 차례**를 익혔습니다. 제약을 먼저 걸고, 노드를 다 넣고, 관계를 잇는 순서였죠. 다음 교안에서는 파일에 담긴 **노드 15,540개와 관계 91,966개**를 통째로 넣습니다. 그 사이에 할 일이 이번 시간입니다.

**같은 사실도 어떻게 모델링하느냐에 따라 나중에 던질 수 있는 질문이 달라집니다.** 무엇을 노드로 두고 무엇을 속성에 담을지, 날짜를 무엇으로 저장할지, 관계 타입을 몇 개로 나눌지, 무엇을 키로 삼을지. 이것을 정하고 나서 적재합니다. 한 번 넣고 나면 바꾸는 값이 비싸집니다.

오늘은 **조선 왕실 계보와 역사 사건**으로 연습합니다. 다음 교안이 적재하는 의료 지식그래프와 도메인이 다른데, 일부러 그렇게 했습니다. 그래프가 관계형 데이터베이스보다 나은 조건은 넷입니다.

- 깊이를 **모르는** 가변 길이 경로
- **경로 자체가 답**인 질문
- 자기 자신을 가리키는 **재귀** 구조
- 여러 대 여러로 겹치는 **다대다**

왕실 계보는 넷을 다 갖습니다. 게다가 **아버지가 둘인 임금**이 실제로 있어서, 관계 타입을 나누는 결정을 피해 갈 수가 없습니다.

<img src="images/joseon_kg_overview.png" width="900">

> **오늘 다루는 것은 이 기록입니다.** 실록에 적힌 임금과 세자, 그 사이의 아버지 자리, 그리고 조선 안팎에서 같은 해에 일어난 일들. 그림 위로 지나가는 실선들이 오늘 만들 그래프입니다. 기록을 그래프로 옮기려면 **무엇을 하나의 것으로 세울지, 무엇으로 이을지**부터 정해야 합니다. 아버지가 둘인 임금을 어떻게 담을지, 음력과 양력이 섞인 날짜를 어느 쪽으로 적을지, 같은 해에 일어난 일들을 어떻게 한자리에 모을지. 그 결정을 하나씩 해 나가는 것이 오늘 할 일입니다.

> **이 교안이 쓰는 사실**: 아래에 나오는 임금·세자·사건과 그 연도는 **실제 기록**입니다.
>
> **딱 한 군데 예외가 있습니다.** 9절에서 밀집 노드를 재현할 때 만드는 `ExDenseEvent` 노드 500개는 **모양만 보려고 만든 더미**라 이름 대신 번호만 붙였습니다. 그 자리에서 다시 알려 드립니다.
>
> **날짜에 주의가 필요합니다.** 조선의 기록은 **음력**이고, 양력으로 환산한 값이 확정된 것과 아닌 것이 섞여 있습니다. 4절이 그 문제 자체를 다룹니다.
>
> **계보는 조각입니다.** 조선의 임금을 다 담지 않고, 임금 아홉 명과 세자 셋만 넣습니다. 그래서 그래프에서 잰 홉 수는 **이 조각 안에서의 홉 수**이지 역사에서의 대수가 아닙니다.

## ⏪ 복습: 지난 교안까지

- **키를 먼저 정했습니다.** 키는 파일에서 겹치지 않는 값이 아니라 **현실에서 같은 것을 같다고 말해 주는 값**으로 골랐습니다.
- **적재 순서**: 제약을 먼저 걸고, 노드를 다 넣고, 관계를 잇습니다. 순서를 어기면 **에러가 아니라 0건**이 됩니다.
- **`MERGE`**: "있으면 그대로, 없으면 만든다". 같은 적재를 다시 돌려도 늘지 않습니다(멱등).
- **관계에 매다는 값**: 두 노드 **사이**의 값은 관계가 들고 있었습니다.

지난 교안은 무엇을 키로 삼을지까지 정하고 넣었습니다. 오늘은 그 앞 단계, **무엇을 노드로 두고 무엇을 관계로 이을지**를 정합니다.

**오늘의 목표**

- [ ] 그래프에서 스키마가 무엇을 뜻하는지 알고, **강제되는 층과 아닌 층**을 가른다.
- [ ] **질문에서** 노드와 관계를 뽑는 절차를 따라간다.
- [ ] 값을 **속성으로 둘지 노드로 뺄지** 두 안을 다 적재해 보고 판단한다.
- [ ] **날짜**를 무엇으로 저장할지 정한다(음력·양력·조인일·발효일).
- [ ] 관계의 **타입·방향·관계 속성**을 정한다.
- [ ] 종류 구분을 **라벨로 둘지 속성으로 둘지** 정한다.
- [ ] 셋 이상이 얽힌 사실을 **중간 노드**로 올린다.
- [ ] **무엇을 키로 삼을지** 정하고 제약으로 잠근다.
- [ ] **밀집 노드**를 알아보고 피한다.

아래 준비 셀 두 개를 실행하세요. **연결 셀은 이해하지 않아도 됩니다.** Neo4j 는 반드시 **실습 전용 DB**에 연결하세요.

> 이 교안은 **작은 실험 노드**만 만듭니다. 두 안을 나란히 적재해 견주고, 절이 끝나면 지웁니다. 레이블도 `ExKing`·`FlatKing`·`NodeKing` 처럼 실험 전용 이름을 써서 다음 교안이 만들 진짜 그래프와 섞이지 않게 합니다.

> 오늘은 **파일을 읽지 않습니다.** 넣을 데이터를 고르는 것이 아니라 넣을 **모양**을 고르는 시간이라, 손으로 적은 몇십 개의 노드면 두 안을 견주기에 충분합니다.

In [1]:
# [제공 코드] Neo4j 연결: 이 셀은 실행만 하세요(내용은 이해하지 않아도 됩니다).
# - .env 의 NEO4J_URI/NEO4J_USER/NEO4J_PASSWORD 로 데이터베이스에 연결합니다.
# - run_cypher("쿼리", 파라미터=값) 가 결과를 dict 리스트로 돌려줍니다. 이 헬퍼로 Cypher 를 실행합니다.
# - 반드시 "실습 전용" 데이터베이스에 연결하세요. 아래 실습이 그래프를 지우고 새로 만듭니다.
import os

from dotenv import load_dotenv
from neo4j import GraphDatabase

# 1) 접속 정보 읽기: .env 의 키=값을 환경변수로 올려 둔다(비밀번호를 코드에 적지 않으려고)
load_dotenv(".env")       # 같은 폴더의 .env
load_dotenv("../.env")    # 정답 폴더에서 실행하는 경우

# os.getenv(키, 기본값): .env 를 못 읽어도 에러가 아니라 이 기본값으로 조용히 넘어간다.
# 그러니 이 셀 마지막 줄에 찍히는 주소가 "실습 전용 DB" 가 맞는지 눈으로 꼭 확인한다
NEO4J_URI = os.getenv("NEO4J_URI", "bolt://localhost:7687")
NEO4J_USER = os.getenv("NEO4J_USER", "neo4j")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD", "neo4j")

# 2) 드라이버: 노트북과 데이터베이스를 잇는 통로를 하나 열어 둔다(노트북이 끝날 때까지 재사용)
driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
driver.verify_connectivity()   # 지금 바로 접속을 시험한다. 여기서 에러가 나면 주소나 계정이 틀린 것


# 3) 공용 헬퍼: 앞으로 모든 Cypher 는 이 함수 하나로 실행한다
def run_cypher(query, **params):
    """Cypher 실행 -> 결과를 dict 리스트로 반환(수업 공용 헬퍼)."""
    # session 은 쿼리를 실행하는 단위. with 블록을 벗어나면 알아서 닫힌다
    with driver.session() as session:
        # record.data() 가 한 행을 dict 로 바꾼다. 그 키는 RETURN 에 적은 별칭이 된다
        return [record.data() for record in session.run(query, **params)]


print("Neo4j 연결:", NEO4J_URI)

Neo4j 연결: bolt://localhost:7687


In [2]:
# [제공 코드] 실습 전용 DB 초기화: 이 데이터베이스를 통째로 비웁니다.
# ⚠️ 가리는 것 없이 **노드·관계·제약을 전부** 지웁니다.
run_cypher("MATCH (n) DETACH DELETE n")   # DETACH 는 노드에 붙은 관계까지 함께 지웁니다
# 제약조건은 노드를 지워도 남습니다. 이름을 조회해 하나씩 DROP 합니다
for _c in run_cypher("SHOW CONSTRAINTS YIELD name RETURN name"):
    run_cypher("DROP CONSTRAINT " + _c["name"] + " IF EXISTS")
print("초기화 완료:", NEO4J_URI, "· 남은 노드:", run_cypher("MATCH (n) RETURN count(n) AS n")[0]["n"])

초기화 완료: bolt://localhost:7687 · 남은 노드: 0


---
# 1. 그래프에서 스키마를 설계한다는 것

관계형 데이터베이스에서 스키마는 **먼저 만들고 나서 데이터를 넣는 틀**입니다. `CREATE TABLE` 로 칸을 정해 두면 없는 칸에는 값을 넣을 수 없습니다. 그래프는 다릅니다. 아무 준비 없이 `MERGE` 한 줄을 실행하면 그 자리에서 노드가 생깁니다. 그러면 그래프에는 스키마가 없는 걸까요?

**있습니다. 다만 데이터베이스가 아니라 사람이 들고 있습니다.**

Neo4j 가 쓰는 자료 모형을 **속성 그래프(labeled property graph)** 라고 부릅니다. 부품은 넷뿐입니다.

| 부품 | 무엇인가 | 오늘의 예 |
|---|---|---|
| **노드** | 하나의 것 | 임금 한 사람, 사건 하나 |
| **라벨** | 그 노드가 어떤 종류인가 | `:ExKing`, `:ExEvent` |
| **관계** | 두 노드를 잇는 것. **타입과 방향**을 갖는다 | `(정조)-[:BIOLOGICAL_CHILD_OF]->(사도세자)` |
| **속성** | 노드와 관계에 붙는 이름=값 | `name: '정조'`, `year: 1776` |

설계란 이 넷을 **무엇으로 채울지 정하는 일**입니다. 그리고 그 결정을 데이터베이스는 대부분 지켜 주지 않습니다.

## 설계는 두 층입니다

| 층 | 무엇이 정하나 | 어기면 |
|---|---|---|
| **논리 스키마** | 설계 문서, 팀의 합의, 적재 코드 | 아무도 막지 않는다. 그래프가 조용히 어긋난다 |
| **강제되는 스키마** | 제약(`IS UNIQUE`·`IS NODE KEY`) | 적재가 에러로 멈춘다 |

그래서 설계는 **문서로 정하고, 지킬 수 있는 부분만 제약으로 잠급니다.** 잠글 수 있는 것은 많지 않습니다. 8절에서 그 자리를 봅니다.

> 스키마가 강제되지 않는 것은 약점이자 강점입니다. 새 종류의 노드가 생겨도 표를 고치고 다시 만들 필요가 없습니다. 지식그래프처럼 **자료가 계속 늘어나는** 일에 그래프를 쓰는 이유가 여기 있습니다.

**그럼 의료 데이터에서는?** 다음 교안이 적재하는 의료 지식그래프는 라벨 5종(`:Compound`·`:Disease`·`:Gene`·`:Symptom`·`:PharmacologicClass`)과 관계 12종으로 설계돼 있습니다. 그런데 데이터베이스가 그 12종만 허용하도록 막고 있지는 않습니다. 강제되는 것은 교안_01 에서 건 **키 제약뿐**이고, 나머지 약속은 **적재 코드가 지킵니다.**

### ✅ 바로 확인 퀴즈

**1.** 그래프 데이터베이스에는 스키마가 없다는 말이 왜 정확하지 않은가요?

<details><summary>정답 보기</summary>

**강제되지 않을 뿐 설계는 있습니다.** 어떤 라벨을 쓸지, 관계 타입을 몇 개로 나눌지, 무엇을 키로 삼을지는 반드시 정해야 하고, 그 결정이 나중에 던질 수 있는 질문을 좌우합니다. 다만 그 결정을 지키는 것은 데이터베이스가 아니라 **설계 문서와 적재 코드**입니다.

</details>

**2.** 라벨 이름을 오타 냈을 때 어떤 신호가 뜨나요?

<details><summary>정답 보기</summary>

**아무 신호도 뜨지 않습니다.** 새 라벨이 하나 생기고 적재는 성공으로 끝납니다. 나중에 라벨별 노드 수를 세거나, 그 라벨로 찾는 쿼리가 예상보다 적게 잡힐 때 드러납니다. 그래서 적재는 항상 **건수로 검증합니다.**

</details>

---
# 2. 질문이 그래프DB를 쓸지와 스키마를 결정합니다

관계형에서는 **자료의 구조**가 표를 정합니다. 사람 자료가 있고 사건 자료가 있으면 표를 그렇게 만듭니다. 그래프는 반대입니다. **던질 질문**이 모양을 정합니다. 같은 사실을 담아도 물을 것이 다르면 좋은 모양이 달라집니다.

그래서 설계의 첫 단계는 그림을 그리는 것이 아니라 **질문을 문장으로 적는 것**입니다. 적어 둔 질문이 두 가지를 차례로 답해 줍니다. 먼저 **이 자료에 그래프DB를 쓸 이유가 있는가**, 그다음 **쓴다면 어떤 모양으로 담을 것인가**입니다.

판단은 그 질문이 이 모양 위에서 **몇 홉**인지 세어 보는 데서 시작합니다.

| 답하고 싶은 질문 | 몇 홉인가 | 이 질문이 부르는 결정 |
|---|---|---|
| 세종의 아버지는 누구인가 | 1홉 | 없다. 조인 한 번이면 된다 |
| 예종은 태조에게서 몇 대 내려온 사람인가 | **가변**(몇 홉인지 미리 모른다) | 그래프DB를 쓸 이유가 여기서 생긴다 |
| 성종의 아버지는 누구인가 | 1홉인데 **타입이 둘**(생부와 양부) | 관계 타입을 나눌 것인가(5절) |
| 1776년에는 무슨 일이 있었나 | 연도를 노드로 올리면 2홉 | 연도를 속성에 둘까 노드로 올릴까(4절) |
| 훈민정음 창제는 몇 대 임금 때인가 | 재위를 노드로 세우면 사건에서 임금까지 2홉 | 재위를 중간 노드로 세울 것인가(7절) |

**첫째, 쓸지 말지가 갈립니다.** 이 목록에 2홉 이상이 하나도 없으면 그래프를 쓸 이유가 약합니다. 1홉짜리 질문은 관계형에서 조인 한 번이면 끝나고, 그쪽이 더 빠르고 더 익숙하고 도구도 많습니다. 그래프는 **관계를 여러 번 건너가는 질문**이 있을 때 값을 합니다.

**둘째, 쓰기로 했다면 그 질문들이 곧 설계서입니다.** 표 오른쪽 칸에 적힌 것이 3절부터 하나씩 마주할 갈림길이고, 어느 쪽을 고를지는 **그 질문에 답하기 편한 쪽**으로 정합니다. 질문을 적어 두지 않으면 고를 근거가 없어 취향으로 정하게 됩니다.

**그럼 의료 데이터에서는?** 의료 그래프의 질문은 이렇습니다. "이 약은 무슨 병에 쓰이나"(1홉), "이 병과 같은 유전자에 얽힌 다른 병은"(2홉), "이 약이 붙는 유전자에 얽힌 병은"(2홉). **2홉짜리가 있어서** 그래프를 씁니다. 1홉 질문만 있었다면 표 두 개와 조인 한 번이 더 나았을 겁니다.

먼저 계보 조각을 한 줄로 이어 놓고 1홉 질문과 가변 홉 질문을 나란히 던져 봅니다.

In [3]:
# 표에서 아버지가 확인되는 임금만 골라 한 줄로 잇는다. 태조에서 예종까지 다섯 사람이다
# 화살표는 자식에서 아버지로 둔다. '누구의 자식이다' 가 자연스럽게 읽히는 쪽이다
run_cypher("""
MERGE (t:ExKing {name: '태조'})
MERGE (tj:ExKing {name: '태종'})
MERGE (sj:ExKing {name: '세종'})
MERGE (sjo:ExKing {name: '세조'})
MERGE (yj:ExKing {name: '예종'})
MERGE (tj)-[:CHILD_OF]->(t)
MERGE (sj)-[:CHILD_OF]->(tj)
MERGE (sjo)-[:CHILD_OF]->(sj)
MERGE (yj)-[:CHILD_OF]->(sjo)
""")
print('사람:', run_cypher("MATCH (k:ExKing) RETURN count(k) AS n")[0]['n'], '명 · 부자 관계:',
      run_cypher("MATCH (:ExKing)-[r:CHILD_OF]->(:ExKing) RETURN count(r) AS n")[0]['n'], '개')

사람: 5 명 · 부자 관계: 4 개


In [4]:
# 질문 1) '예종의 아버지는?' 관계를 한 번만 건너간다. 관계형이었다면 조인 한 번이다
print('예종의 아버지:', run_cypher("MATCH (:ExKing {name: '예종'})-[:CHILD_OF]->(f:ExKing) "
                                "RETURN f.name AS 아버지"))

예종의 아버지: [{'아버지': '세조'}]


In [ ]:
# 질문 2) '예종은 태조에게서 몇 홉인가?' * 는 '몇 번인지 모르니 닿을 때까지' 라는 뜻이다
# [x IN nodes(p) | x.name] 은 경로에 놓인 노드를 차례로 꺼내 이름만 모은 것이다
for row in run_cypher("MATCH p = (:ExKing {name: '예종'})-[:CHILD_OF*]->(:ExKing {name: '태조'}) "
                      "RETURN length(p) AS 홉수, [x IN nodes(p) | x.name] AS 경로"):
    print(row)

> 4홉이고, 지나온 다섯 사람이 함께 나왔습니다. 여기서 두 가지가 드러납니다.
> 1. **몇 홉인지 쿼리에 적지 않았습니다.** `*` 하나로 끝났습니다. 관계형에서 같은 것을 물으려면 재귀 질의(`WITH RECURSIVE`)를 씁니다. 됩니다. 다만 쿼리가 길어지고, 깊이가 답의 일부인 질문이 몇 개만 쌓여도 다루기 어려워집니다.
> 2. **경로 자체가 답의 일부입니다.** "몇 대인가"뿐 아니라 "누구를 거쳐서인가"를 함께 돌려줍니다. 관계형에서는 거쳐 온 행을 따로 모아야 합니다.

## 설계 절차

질문을 적었으면 다음 순서로 모양을 뽑습니다. 크게 세 덩어리입니다.

<img src="images/schema_design_steps.png" width="880">

**밑그림**

1. **질문 정의**: 답하고 싶은 질문을 문장으로 적는다(지금 이 절).
2. **명사에서 노드**: 질문에 나오는 명사를 뽑아 노드 후보로 삼는다(바로 아래).
3. **동사에서 관계**: 질문에 나오는 동사를 뽑아 관계 타입 이름을 짓는다(바로 아래).

**다듬는 고리**

4. **속성 배치**: 남은 값을 노드 속성·관계 속성·라벨 중 어디에 둘지 정한다(3·4·5·6·7절).
5. **다이어그램**: 그 모양을 그림으로 그려 팀과 맞춘다.
6. **소량 샘플 적재**: 몇십 건만 실제로 넣어 본다.
7. **실제 질의로 검증**: 1번에서 적은 질문을 그 모양 위에서 직접 던져 본다.
8. **모델 수정**: 쿼리가 매끄럽지 않으면 4번으로 돌아간다. 조건을 **매번 손으로 덧붙여야 하거나** 묻고 싶은 것을 **담을 자리가 없으면** 모양이 틀린 것이다.

**본 적재**

9. **제약과 인덱스 생성**: 키를 정해 잠근다(8절).
10. **전체 적재**: 파일을 통째로 넣는다(교안_03).

## 이름 짓는 규칙

이름은 취향이 아니라 **약속**입니다. 팀이 같은 규칙을 쓰지 않으면 `:King` 과 `:Kings` 가 따로 생깁니다.

| 대상 | 규칙 | 좋은 예 | 나쁜 예 |
|---|---|---|---|
| 노드 라벨 | 파스칼 표기, **단수** | `King`, `Event` | `kings`, `king_table` |
| 관계 타입 | 대문자 스네이크, **동사구** | `CHILD_OF`, `HAPPENED_IN` | `HAS`, `RELATED_TO` |
| 속성 | 소문자 스네이크 | `reign_start`, `signed_on` | `ReignStart`, `date2` |
| 방향 | 읽어서 문장이 되는 쪽 | `(정조)-[:CHILD_OF]->(사도세자)` | 뜻과 반대로 저장 |

## 질문에서 노드와 관계를 뽑습니다

절차의 2번과 3번을 지금 해 봅니다. 방법은 단순합니다. **앞에서 적은 질문 문장에서 명사와 동사를 뽑습니다.**

| 질문에 나온 명사 | 노드 후보 | 지금 세울까요 |
|---|---|---|
| 세종·예종·성종 같은 사람 | `:King` | 세운다 |
| 훈민정음 창제 같은 일 | `:Event` | 세운다 |
| 1776년 같은 해 | `:Year` | **후보로만 둔다**(4절에서 정한다) |
| 재위 | `:Reign` | **후보로만 둔다**(3절에서 정한다) |

\

| 질문에 나온 동사 | 관계 후보 | 방향 |
|---|---|---|
| ~의 아버지이다 | `[:CHILD_OF]` | 자식에서 아버지로 |

관계 타입 이름은 위 규칙표대로 **대문자 스네이크의 동사구**로 지었습니다. `(예종)-[:CHILD_OF]->(세조)` 는 읽으면 문장이 됩니다.

**명사라고 다 노드가 되지는 않습니다.** 이름도 태어난 해도 명사지만, 그 값을 거쳐 다른 데로 건너갈 일이 없어 속성으로 갑니다. 첫 밑그림에는 **확실한 것만** 세우고 애매한 것은 후보로 적어 둡니다. 그 판정이 4번 속성 배치이고, 재위는 3절에서 연도는 4절에서 정합니다.

임금은 위에서 이미 이었으니 사건만 얹으면 밑그림이 됩니다. 연도는 아직 노드가 아니니 사건의 **속성 칸**에 적어 둡니다.

In [6]:
# 밑그림대로 사건을 얹는다. 임금 다섯과 CHILD_OF 는 위에서 이미 만들었다
# 연도는 아직 후보라서 노드로 세우지 않고 사건의 속성 칸에 적어 둔다
run_cypher("""
MERGE (a:ExEvent {name: '훈민정음 창제'}) SET a.year = 1443
MERGE (b:ExEvent {name: '계해약조'})      SET b.year = 1443
""")
print('사건:', run_cypher("MATCH (e:ExEvent) RETURN count(e) AS n")[0]['n'], '개')

사건: 2 개


In [7]:
# 질문 A) '1443년에는 무슨 일이 있었나' 연도가 속성이라 값을 견주어 찾는다
for row in run_cypher("MATCH (e:ExEvent) WHERE e.year = 1443 RETURN e.name AS 사건"):
    print(row)

{'사건': '훈민정음 창제'}
{'사건': '계해약조'}


In [8]:
# 질문 B) '훈민정음 창제는 몇 대 임금 때인가' 밑그림 위에서 그대로 던져 본다
# 화살표 없는 -- 는 방향을 가리지 않고 이어져 있기만 하면 걸린다는 뜻이다
rows = run_cypher("MATCH (:ExEvent {name: '훈민정음 창제'})--(k:ExKing) RETURN k.name AS 임금")
print('찾은 임금:', len(rows), '명')

찾은 임금: 0 명


> 한 질문은 답했고 한 질문은 **못 했습니다.** 에러가 아니라 **빈 결과**입니다. 답이 틀린 것이 아니라 물을 자리가 아예 없는 것입니다.

못 한 이유는 **재위**를 아직 아무 데도 두지 않았기 때문입니다. 훈민정음 창제는 세종이라는 사람에게 바로 붙는 사실이 아니라 **세종의 재위 중에** 일어난 일입니다. 재위를 임금의 속성 칸에 둘지 노드로 세울지 정하지 않았으니, 사건을 이어 붙일 자리도 아직 없습니다.

연도도 지금은 사건의 속성 칸에 얹어 뒀을 뿐입니다. 그대로 둘지 노드로 올릴지는 4절에서 정합니다.

**밑그림은 처음부터 완벽할 필요가 없습니다.** 답이 안 나오는 질문이 다듬을 자리를 알려 줍니다. 절차 8번이 4번으로 돌아가는 고리인 이유가 이것입니다. 3절부터는 그 고리를 한 바퀴씩 돌면서 남은 값을 어디에 둘지 하나씩 정합니다.

### ✅ 바로 확인 퀴즈

**1.** 던질 질문이 전부 1홉이라면 그래프 데이터베이스를 쓸 이유가 있을까요?

<details><summary>정답 보기</summary>

약합니다. 1홉 질문은 관계형에서 **조인 한 번**이면 끝나고, 관계형 쪽이 더 빠르고 도구도 많습니다. 그래프는 **관계를 여러 번 건너가는 질문**, 특히 **몇 번 건너갈지 미리 모르는 질문**이 있을 때 값을 합니다.

</details>

**2.** `-[:CHILD_OF*]->` 의 `*` 를 빼고 `-[:CHILD_OF]->` 로 쓰면 위 질문의 답이 어떻게 달라지나요?

<details><summary>정답 보기</summary>

**한 홉만** 건너가므로 예종에서 태조로 가는 경로를 찾지 못하고 **결과가 0건**이 됩니다. 에러가 아니라 빈 결과라 눈치채기 어렵습니다. 몇 홉인지 모르는 질문에는 `*` 를 씁니다.

</details>

**3.** 관계 타입을 `:HAS_FATHER` 대신 `:CHILD_OF` 로 지으면 무엇이 좋아지나요?

<details><summary>정답 보기</summary>

**방향과 함께 읽으면 문장이 됩니다.** `(예종)-[:CHILD_OF]->(세조)` 는 "예종은 세조의 자식이다"로 읽힙니다. `:HAS_FATHER` 도 뜻은 통하지만 `HAS` 로 시작하는 이름은 무엇을 가졌다는 것인지 흐려지기 쉽고, 결국 온갖 뜻이 한 타입에 쌓입니다(5절).

</details>

**4.** "이 약은 무슨 병에 쓰이나"라는 질문에서 노드 후보와 관계 타입 이름을 뽑아 보세요.

<details><summary>정답 보기</summary>

명사 둘에서 노드 후보 `:Compound`(약물)와 `:Disease`(질병)가 나오고, 동사 "쓰인다"에서 관계 타입 `[:TREATS]` 가 나옵니다. 방향은 읽어서 문장이 되는 쪽이라 **약에서 병으로** 둡니다. 다음 교안이 적재하는 의료 그래프가 실제로 이 이름을 씁니다.

</details>

In [ ]:
# 2절 실험 노드를 지운다
run_cypher("MATCH (n) WHERE n:ExKing OR n:ExEvent DETACH DELETE n")
print('남은 2절 노드:', run_cypher("MATCH (n) WHERE n:ExKing OR n:ExEvent "
                                "RETURN count(n) AS n")[0]['n'], '개')

---
# 3. 무엇을 노드로 두고 무엇을 속성으로 둘까요?

2절 밑그림에서 임금과 사건은 노드가 됐고, 남은 것이 **재위**입니다. 절차의 4번, 남은 값을 어디에 둘지 정하는 일을 여기서 시작합니다.

임금 한 사람에 대해 우리가 아는 것은 이름, 태어난 해, 죽은 해, 그리고 **몇 대 임금으로 언제부터 언제까지 왕위에 있었는가**입니다. 앞의 셋은 그 사람을 설명하는 값이라 속성이 자연스럽습니다. 마지막 것, **재위**는 어떨까요?

겉보기에는 이것도 그 사람을 설명하는 값입니다. 칸 세 개(`order_no`·`reign_start`·`reign_end`)를 만들면 될 것 같습니다. 정답이 하나인 문제가 아닙니다. 그래서 **두 안을 다 적재해 같은 질문을 던져 봅니다.**

- **안 A**: 재위 정보를 임금의 속성 칸에 둔다(`:FlatKing`).
- **안 B**: 재위를 노드로 빼고 관계로 잇는다(`:NodeKing` -> `:ExReign`).

In [9]:
# 안 A: 표를 그대로 옮긴 모양이다. 임금 한 사람이 한 줄이고 재위는 그 줄의 칸이다
run_cypher("""
MERGE (a:FlatKing {name: '세종'}) SET a.order_no = 4, a.reign_start = 1418, a.reign_end = 1450
MERGE (b:FlatKing {name: '문종'}) SET b.order_no = 5, b.reign_start = 1450, b.reign_end = 1452
MERGE (c:FlatKing {name: '세조'}) SET c.order_no = 7, c.reign_start = 1455, c.reign_end = 1468
MERGE (d:FlatKing {name: '예종'}) SET d.order_no = 8, d.reign_start = 1468, d.reign_end = 1469
""")
print('안 A 임금:', run_cypher("MATCH (k:FlatKing) RETURN count(k) AS n")[0]['n'], '명')

안 A 임금: 4 명


In [10]:
# 안 B: 재위를 노드로 빼고 관계로 잇는다. 같은 사실, 다른 모양
# 재위 노드의 키는 대수다. '4대 재위' 는 하나뿐이라 MERGE 로 중복 없이 만들어진다
run_cypher("""
MERGE (a:NodeKing {name: '세종'})
MERGE (b:NodeKing {name: '문종'})
MERGE (c:NodeKing {name: '세조'})
MERGE (d:NodeKing {name: '예종'})
MERGE (r4:ExReign {order_no: 4}) SET r4.start_year = 1418, r4.end_year = 1450
MERGE (r5:ExReign {order_no: 5}) SET r5.start_year = 1450, r5.end_year = 1452
MERGE (r7:ExReign {order_no: 7}) SET r7.start_year = 1455, r7.end_year = 1468
MERGE (r8:ExReign {order_no: 8}) SET r8.start_year = 1468, r8.end_year = 1469
MERGE (a)-[:REIGNED]->(r4)
MERGE (b)-[:REIGNED]->(r5)
MERGE (c)-[:REIGNED]->(r7)
MERGE (d)-[:REIGNED]->(r8)
""")
print('안 B 재위 노드:', run_cypher("MATCH (r:ExReign) RETURN count(r) AS n")[0]['n'], '개')

안 B 재위 노드: 4 개


In [11]:
# 질문 1) '세종은 언제부터 왕위에 있었나' 두 안 모두 답한다. 다른 것은 쿼리의 모양이다
print('안 A:', run_cypher("MATCH (k:FlatKing {name: '세종'}) RETURN k.reign_start AS 시작"))
# 안 B 는 임금에서 재위 노드로 한 번 건너간 뒤에야 값이 나온다. 한 홉 더 든다
print('안 B:', run_cypher("MATCH (:NodeKing {name: '세종'})-[:REIGNED]->(r:ExReign) "
                        "RETURN r.start_year AS 시작"))

안 A: [{'시작': 1418}]
안 B: [{'시작': 1418}]


In [12]:
# 질문 2) '1450년에 왕위에 있던 사람은?' 범위 비교라 값을 직접 들고 있는 쪽이 곧바로 된다
# collect 앞에 ORDER BY 를 두어야 순서가 정해진다. 두지 않으면 실행할 때마다 순서가 달라질 수 있다
print('안 A:', run_cypher("MATCH (k:FlatKing) WHERE k.reign_start <= 1450 AND 1450 <= k.reign_end "
                        "WITH k ORDER BY k.order_no RETURN collect(k.name) AS 임금"))
# 안 B 도 되지만 재위 노드까지 건너간 뒤 그 노드의 값을 견줘야 한다
print('안 B:', run_cypher("MATCH (k:NodeKing)-[:REIGNED]->(r:ExReign) "
                        "WHERE r.start_year <= 1450 AND 1450 <= r.end_year "
                        "WITH k, r ORDER BY r.order_no RETURN collect(k.name) AS 임금"))

안 A: [{'임금': ['세종', '문종']}]
안 B: [{'임금': ['세종', '문종']}]


> 두 질문 모두 답이 같습니다. 그러면 안 A 로 충분해 보입니다. 실제로 **범위로 자르고 크기를 견주는 일만** 한다면 안 A 가 낫습니다. 값이 노드에 붙어 있어 관계를 건너갈 필요가 없으니까요.

차이는 **표에 담기지 않는 사람**이 들어올 때 드러납니다. 의경세자(덕종)는 세조의 장남인데 왕위에 오르지 못하고 죽어 나중에 추존됐습니다. **재위가 없는 사람**입니다.

In [13]:
# 질문 3) 왕위에 오른 적이 없는 사람이 섞이면? 의경세자는 추존이라 재위 자체가 없다
run_cypher("""
MERGE (:FlatKing {name: '의경세자'})
MERGE (:NodeKing {name: '의경세자'})
""")
# 안 A: 재위 칸이 빈 채로 남는다. '재위한 사람만' 세려면 빈 칸을 조건으로 걸어야 한다
print('안 A 전체:', run_cypher("MATCH (k:FlatKing) RETURN count(k) AS n")[0]['n'],
      '· 재위 있는 사람:', run_cypher("MATCH (k:FlatKing) WHERE k.reign_start IS NOT NULL "
                                   "RETURN count(k) AS n")[0]['n'])
# 안 B: 재위 노드가 없을 뿐이다. 패턴 자체가 조건이 되어 빈 칸을 다룰 일이 없다
print('안 B 전체:', run_cypher("MATCH (k:NodeKing) RETURN count(k) AS n")[0]['n'],
      '· 재위 있는 사람:', run_cypher("MATCH (k:NodeKing)-[:REIGNED]->(:ExReign) "
                                   "RETURN count(k) AS n")[0]['n'])

안 A 전체: 5 · 재위 있는 사람: 4
안 B 전체: 5 · 재위 있는 사람: 4


> 두 안 모두 다섯 명 중 네 명이 재위를 갖습니다. 답은 같은데 **쓰는 사람이 기억해야 할 것**이 다릅니다. 안 A 에서는 `reign_start IS NOT NULL` 을 매번 붙여야 하고, 빠뜨리면 재위가 없는 사람이 결과에 섞입니다. 안 B 에서는 `-[:REIGNED]->` 를 쓰는 순간 재위가 있는 사람만 걸립니다. **없는 것을 `null` 로 표현하지 않아도 됩니다.**

그리고 안 B 에는 안 A 가 아예 할 수 없는 일이 하나 더 있습니다. **재위 노드에 다른 것을 이어 붙이는 일**입니다. "훈민정음 창제는 어느 재위 중에 일어났나"를 담으려면 이을 자리가 필요한데, 안 A 에는 그 자리가 없습니다. 7절에서 이어서 봅니다.

## 판단 기준: 속성일까요, 노드일까요?

| 이런 값이면 | 어디에 두나 | 오늘의 예 |
|---|---|---|
| 그 개체를 **설명하기만** 하고 다른 것과 이어지지 않는다 | **속성** | 임금의 이름, 태어난 해 |
| 그 값 **자체에 붙일 것**이 있다 | **노드** | 재위(대수와 기간을 함께 갖는다) |
| 그 값을 **거쳐 다른 것으로 건너갈** 일이 있다 | **노드** | 재위(그 시기의 사건으로 건너간다) |
| 값이 **없는 경우가 흔하다** | **노드** | 재위(추존은 재위가 없다) |
| 같은 값이 **여러 개체에 반복**되고 그 값으로 묶어 세는 일이 잦다 | **노드** | 연도(4절에서 봅니다) |
| **범위로 자르고 크기를 견주기만** 한다 | **속성** | 재위 시작 연도 |
| 값이 개체마다 거의 다르고 반복되지 않는다 | **속성** | 이름 |

가장 실용적인 기준은 **"이 값을 거쳐 다른 것으로 건너갈 일이 있는가"** 입니다. 노드로 빼 두면 그 값이 **경로의 징검다리**가 됩니다. 속성은 그렇게 못 씁니다.

> 그리고 **둘 다 두는 것**도 흔한 답입니다. 재위를 노드로 빼면서 임금에도 `reign_start` 를 남겨 두면, 묶는 일은 노드가 견주는 일은 속성이 맡습니다. 4절에서 날짜로 다시 봅니다.

**그럼 의료 데이터에서는?** 다음 교안이 적재하는 자료에서 **약효분류**는 약의 속성 칸이 아니라 `:PharmacologicClass` **노드**입니다. 갈래 자체에 분류 코드가 붙고, 한 약이 갈래를 여럿 가지며, 그 갈래를 거쳐 다른 약으로 건너갈 일이 있어서입니다. 위 표의 세 줄이 한꺼번에 노드를 가리킨 경우입니다.

### ✅ 바로 확인 퀴즈

**1.** 재위를 임금의 속성으로만 두면 무엇이 불편한가요? 두 가지를 드세요.

<details><summary>정답 보기</summary>

1. **재위 자체에 다른 것을 이어 붙일 수 없습니다.** "그 재위 중에 일어난 사건"을 담을 자리가 없습니다.
2. **재위가 없는 사람을 빈 칸으로 표현해야 합니다.** "재위한 사람만"을 물을 때마다 `IS NOT NULL` 조건을 붙여야 하고, 빠뜨리면 조용히 섞여 들어옵니다.

</details>

**2.** 반대로 노드로 빼면 **안 되는** 값의 예를 하나 들어 보세요.

<details><summary>정답 보기</summary>

임금의 **이름**처럼 개체마다 거의 다르고 반복되지 않는 값입니다. 노드로 빼면 이름 하나에 노드 하나가 생겨 그래프가 두 배로 커지는데, 그 노드를 거쳐 갈 곳이 없어 얻는 것이 없습니다.

</details>

**3.** 안 B 에서 재위 노드의 키를 이름(`{name: '세종의 재위'}`)이 아니라 대수(`{order_no: 4}`)로 삼은 이유는?

<details><summary>정답 보기</summary>

대수는 **조선 안에서 겹치지 않는 값**이라 같은 재위를 두 번 만들지 않게 해 줍니다. 이름으로 삼으면 적는 사람마다 표기가 달라져("세종의 재위", "세종 재위") 같은 재위가 여러 노드로 갈라집니다. 8절에서 키 고르기를 따로 봅니다.

</details>

In [ ]:
# 3절 실험 노드를 지운다
run_cypher("MATCH (n) WHERE n:FlatKing OR n:NodeKing OR n:ExReign DETACH DELETE n")
print('남은 3절 노드:', run_cypher("MATCH (n) WHERE n:FlatKing OR n:NodeKing OR n:ExReign "
                                "RETURN count(n) AS n")[0]['n'], '개')

---
# 4. 날짜: 무엇을 저장할 것인가

"이 사건은 언제 일어났나"는 쉬워 보입니다. 날짜 칸 하나면 될 것 같습니다. 실제 기록을 옮겨 보면 그렇지 않습니다.

| 사건 | 기록에 남은 '언제' |
|---|---|
| 훈민정음 반포 | 음력 1446년 9월 10일. **양력 환산이 확정이 아니다**(그레고리력 10월 9일, 율리우스력 9월 30일) |
| 임진왜란 발발 | 음력 1592년 4월 13일 = 양력 1592년 5월 23일 |
| 훈민정음 창제 | 음력 1443년 12월. **일이 없다** |
| 병자호란 | 1636년부터 1637년까지. **한 날이 아니라 기간** |
| 강화도조약 | 1876년 2월 27일 서명식(조인 자체는 2월 26일에 끝났다) |
| 한일병합조약 | **조인 1910년 8월 22일, 발효 8월 29일** |
| 조선 건국 | 음력 1392년 8월 13일. **국호를 조선으로 정한 것은 1393년 3월 28일로 따로다** |

날짜 칸 하나에 이걸 다 담으면 **무엇을 담았는지 아무도 모릅니다.** 순서대로 셋을 정합니다.

1. 무엇이 그 사건의 '언제'인가
2. 얼마나 정밀하게 아는가
3. 속성에 둘 것인가 노드로 올릴 것인가

## 4-1. 무엇이 그 사건의 '언제'인가

한일병합조약은 조인과 발효 사이가 **일주일**입니다. 이 그래프에 `date` 라는 칸 하나만 두고 둘 중 하나를 넣으면, 나중에 그 값을 쓰는 사람은 자기가 무엇을 쓰고 있는지 모릅니다. 속성 이름을 갈라 봅시다.

In [ ]:
# 조인일과 발효일을 각각 다른 이름의 속성으로 둔다. 속성 이름이 곧 '이 값이 무엇인가' 의 정의다
run_cypher("""
MERGE (e:ExEvent {name: '한일병합조약'})
SET e.signed_on = date('1910-08-22'), e.effective_from = date('1910-08-29')
""")
# 1910-08-25 라는 같은 시점을 두 속성으로 각각 물어 본다. 답이 갈린다
print(run_cypher("MATCH (e:ExEvent {name: '한일병합조약'}) "
                 "RETURN e.signed_on <= date($on) AS 조인됐나, "
                 "       e.effective_from <= date($on) AS 발효됐나", on='1910-08-25'))

> 같은 날을 두고 **조인은 끝났고 발효는 아직**입니다. "1910년 8월 25일에 조약이 효력을 갖고 있었나"의 답이 어느 칸을 보느냐로 갈립니다. 칸 이름이 `date` 였다면 이 물음에 답할 수 없습니다.

## 4-2. 얼마나 정밀하게 아는가

`date('1443-12-01')` 이라고 적으면 데이터베이스는 만족합니다. 날짜 자료형은 **일까지** 요구하니까요. 그런데 기록에 남은 것은 음력 1443년 **12월**뿐입니다. 없는 정보를 지어낸 셈이고, 나중에 읽는 사람은 12월 1일에 일어난 일로 읽습니다. **아는 만큼만 적고, 어디까지 아는지를 함께 적습니다.**

In [ ]:
# 정밀도가 다른 세 사건을 각기 아는 만큼만 담는다. precision 이 '어디까지 믿어도 되는가' 를 말해 준다
run_cypher("""
MERGE (a:ExEvent {name: '임진왜란 발발'})
SET a.solar_date = date('1592-05-23'), a.lunar_date = '1592-04-13', a.precision = '일'
MERGE (b:ExEvent {name: '훈민정음 창제'})
SET b.lunar_year = 1443, b.lunar_month = 12, b.precision = '월'
MERGE (c:ExEvent {name: '병자호란'})
SET c.start_year = 1636, c.end_year = 1637, c.precision = '연'
""")
for row in run_cypher("MATCH (e:ExEvent) WHERE e.precision IS NOT NULL "
                      "RETURN e.name AS 사건, e.precision AS 정밀도 ORDER BY 사건"):
    print(row)

> 세 사건의 정밀도가 일·월·연으로 제각각입니다. 임진왜란만 `solar_date` 에 날짜 자료형이 들어갔고, 나머지는 정수 속성으로 아는 만큼만 담았습니다. 음력 값은 `lunar_date` 라는 **글자 속성**으로 따로 남겼습니다. 음력 1592년 4월 13일을 `date('1592-04-13')` 으로 적으면 양력 날짜로 오해받습니다.

**속성 이름 짓기가 곧 정의 쓰기입니다.**

| 이런 이름 | 왜 나쁜가 | 대신 |
|---|---|---|
| `date` | 무슨 날짜인지 말하지 않는다 | `signed_on`, `effective_from`, `solar_date` |
| `year` 하나만 | 시작인지 끝인지 모른다 | `start_year`, `end_year` |
| 음력 값을 `date()` 로 | 양력으로 오해받는다 | `lunar_date` 를 글자로 따로 |

## 4-3. 속성에 둘까요, 노드로 올릴까요

이제 3절의 판단 기준을 날짜에 대 봅니다. 날짜는 **같은 값이 여러 사건에 반복**되고, 그 값 **자체에 붙일 것**이 있고(무슨 해인가, 어떤 시대인가), 그 값을 **거쳐 다른 사건으로 건너갈** 일이 있습니다. 표의 여러 줄이 노드를 가리킵니다.

연도를 노드로 올려 두 해를 만들어 봅시다. **1776년**과 **1894년**은 여러 일이 겹친 해입니다.

In [ ]:
# 연도를 노드로 올린다. 그러면 '같은 해에 있었던 일' 이 값 비교가 아니라 공유 패턴이 된다
run_cypher("""
MERGE (y1:ExYear {value: 1776})
MERGE (y2:ExYear {value: 1894})
MERGE (a:ExEvent {name: '정조 즉위'})      MERGE (a)-[:HAPPENED_IN]->(y1)
MERGE (b:ExEvent {name: '미국 독립선언'})  MERGE (b)-[:HAPPENED_IN]->(y1)
MERGE (c:ExEvent {name: '청일전쟁 발발'})  MERGE (c)-[:HAPPENED_IN]->(y2)
MERGE (d:ExEvent {name: '갑오개혁'})       MERGE (d)-[:HAPPENED_IN]->(y2)
MERGE (e:ExEvent {name: '동학농민운동'})   MERGE (e)-[:HAPPENED_IN]->(y2)
""")
print('연도 노드:', run_cypher("MATCH (y:ExYear) RETURN count(y) AS n")[0]['n'], '개 · 연결된 사건:',
      run_cypher("MATCH (:ExEvent)-[r:HAPPENED_IN]->(:ExYear) RETURN count(r) AS n")[0]['n'], '건')

In [ ]:
# 가운데 연도 노드를 함께 가리키는 두 사건을 찾는다. 값을 견주는 조건이 아예 없는 것이 요점이다
# 이름에 순서를 주지 않으면 같은 쌍이 앞뒤를 바꿔 두 번 세어진다
for row in run_cypher("MATCH (a:ExEvent)-[:HAPPENED_IN]->(y:ExYear)<-[:HAPPENED_IN]-(b:ExEvent) "
                      "WHERE a.name < b.name "
                      "RETURN y.value AS 해, a.name AS 사건1, b.name AS 사건2 "
                      "ORDER BY 해, 사건1, 사건2"):
    print(row)

> 1776년에 정조가 즉위한 것과 미국이 독립을 선언한 것이 **같은 해**입니다. 1894년에는 셋이 겹쳐 쌍이 셋 나왔습니다. 날짜를 속성으로만 뒀다면 `a.year = b.year` 라는 **값 비교**를 써야 했습니다. 노드로 올리면 **가운데 노드를 함께 가리키는 모양** 자체가 조건이 됩니다.

이게 지식그래프에서 연도 노드가 자주 쓰이는 이유입니다. 서로 무관해 보이는 사건이 **같은 해**라는 고리로 이어지고, 그 고리를 따라 건너갈 수 있습니다.

> 대신 값을 **견주고 자르는 일**은 속성 쪽이 곧바로 됩니다(`WHERE e.solar_date >= date('1800-01-01')`). 그래서 실무에서는 **둘 다 둡니다.** 연도 노드를 만들어 잇고, 사건에도 날짜 속성을 그대로 남깁니다. 일어난 날은 한 번 정해지면 바뀌지 않는 값이라 두 곳에 있어도 어긋나지 않습니다.

> 그리고 값이 붙습니다. 연도를 노드로 올리면 **사건 한 건마다 관계 한 개**가 생깁니다. 사건이 100만 건이면 관계도 100만 개입니다. 게다가 한 연도에 사건이 몰리면 그 노드 자체가 골칫거리가 됩니다. 9절에서 그 값을 직접 세어 봅니다.

| 이럴 때 | 어떻게 하나 |
|---|---|
| 같은 날·같은 해 것끼리 이어 보는 일이 잦다 | 노드로 올린다 |
| 날짜 자체에 값을 붙인다 | 노드로 올린다 |
| 그 날짜를 거쳐 다른 것으로 건너간다 | 노드로 올린다 |
| 범위로 자르고 기간을 재기만 한다 | 속성으로 충분하다 |
| 건수가 아주 많은데 날짜로 묶어 볼 일이 드물다 | 속성만 둔다(관계 비용이 아깝다) |

**그럼 의료 데이터에서는?** 의료 자료에서 "언제"는 진단일·처방일·문헌 보고일이 각각 다릅니다. 다음 교안이 쓰는 자료는 **2016년에 정리된 것**이라, 그 안의 `TREATS` 관계는 "그때까지 문헌에 정리돼 있다"는 뜻이지 "효능이 입증된 날"이 아닙니다. 그 구분을 속성 이름과 설명에 적어 두지 않으면 읽는 쪽이 다른 뜻으로 읽습니다. **날짜 모델링은 사실을 왜곡하지 않기 위한 장치이기도 합니다.**

### ✅ 바로 확인 퀴즈

**1.** 한일병합조약처럼 조인일과 발효일이 다른 사건을 날짜 칸 하나(`date`)에 담으면 어떤 문제가 생기나요?

<details><summary>정답 보기</summary>

그 칸에 무엇이 들어 있는지 **알 수 없습니다.** "이 날 조약이 효력을 갖고 있었나"를 물으면 답이 갈리는데, 어느 쪽으로 답했는지 데이터가 말해 주지 않습니다. 속성 이름을 `signed_on`·`effective_from` 으로 갈라 두면 이름 자체가 정의가 됩니다.

</details>

**2.** 음력 1443년 12월에 일어난 일을 `date('1443-12-01')` 로 적으면 무엇이 문제인가요?

<details><summary>정답 보기</summary>

두 가지입니다. **없는 정보(1일)를 지어냈고**, **음력 값을 양력으로 오해받게** 만들었습니다. 아는 만큼만 정수 속성(`lunar_year`·`lunar_month`)으로 담고, 어디까지 아는지를 `precision` 같은 칸에 함께 적습니다.

</details>

**3.** "같은 해에 일어난 두 사건"을 찾을 때, 연도를 속성으로 둔 안과 노드로 올린 안은 쿼리 모양이 어떻게 다른가요?

<details><summary>정답 보기</summary>

속성으로 두면 사건을 **둘 다 잡아 놓고 값이 같은지 견주는** 조건(`a.year = b.year`)이 필요합니다. 노드로 올리면 두 사건이 **가운데 연도 노드 하나를 함께 가리키는 패턴** 자체가 조건이 되어, 값을 견주는 조건이 사라집니다.

</details>

In [ ]:
# 4절 실험 노드를 지운다
run_cypher("MATCH (n) WHERE n:ExEvent OR n:ExYear DETACH DELETE n")
print('남은 4절 노드:', run_cypher("MATCH (n) WHERE n:ExEvent OR n:ExYear "
                                "RETURN count(n) AS n")[0]['n'], '개')

---
# 5. 관계: 타입·방향·관계 속성

노드를 정했으면 관계입니다. 정할 것은 셋입니다. **타입을 몇 개로 나눌지**, **방향을 어디로 둘지**, **연결 자체에 붙는 값을 어디에 둘지**.

## 5-1. 뜻이 다르면 타입을 나눕니다

조선 왕실에는 이 결정을 피할 수 없게 만드는 사실이 있습니다.

> **정조는 생부가 사도세자인데, 영조의 명으로 효장세자의 양자로 입적되어 왕위계승권을 유지했습니다.** 정조는 즉위식 당일 "사도세자의 아들이긴 하지만 영조께서 효장세자의 아들로 만들어 놓았으니 그것을 그대로 지켜야 한다"고 천명했습니다.
>
> **성종도 같습니다.** 생부는 의경세자(덕종)이고, 숙부인 예종의 양자 자격으로 즉위했습니다.

그러니 "아버지"를 관계 하나로 둘 수 없습니다. 하나로 두면 **아버지가 둘인 사람**이 생기고, 둘 중 어느 쪽인지 구분이 사라집니다. 관계 타입을 둘로 나눕니다.

- `BIOLOGICAL_CHILD_OF`: 혈통상 자식이다.
- `ADOPTED_CHILD_OF`: 입적으로 자식이 됐다.

In [ ]:
# 혈통과 입적을 다른 관계 타입으로 나눠 저장한다. 성종과 정조 두 줄기를 담는다
run_cypher("""
MERGE (tj:ExPerson {name: '태종'})
MERGE (sj:ExPerson {name: '세종'})
MERGE (sjo:ExPerson {name: '세조'})
MERGE (uk:ExPerson {name: '의경세자'})
MERGE (yj:ExPerson {name: '예종'})
MERGE (sk:ExPerson {name: '성종'})
MERGE (yo:ExPerson {name: '영조'})
MERGE (hj:ExPerson {name: '효장세자'})
MERGE (sd:ExPerson {name: '사도세자'})
MERGE (jj:ExPerson {name: '정조'})
MERGE (sj)-[:BIOLOGICAL_CHILD_OF]->(tj)
MERGE (sjo)-[:BIOLOGICAL_CHILD_OF]->(sj)
MERGE (uk)-[:BIOLOGICAL_CHILD_OF]->(sjo)
MERGE (yj)-[:BIOLOGICAL_CHILD_OF]->(sjo)
MERGE (sk)-[:BIOLOGICAL_CHILD_OF]->(uk)
MERGE (sk)-[:ADOPTED_CHILD_OF]->(yj)
MERGE (hj)-[:BIOLOGICAL_CHILD_OF]->(yo)
MERGE (sd)-[:BIOLOGICAL_CHILD_OF]->(yo)
MERGE (jj)-[:BIOLOGICAL_CHILD_OF]->(sd)
MERGE (jj)-[:ADOPTED_CHILD_OF]->(hj)
""")
print('사람:', run_cypher("MATCH (p:ExPerson) RETURN count(p) AS n")[0]['n'], '명 · 혈통 관계:',
      run_cypher("MATCH (:ExPerson)-[r:BIOLOGICAL_CHILD_OF]->(:ExPerson) "
                 "RETURN count(r) AS n")[0]['n'], '개 · 입적 관계:',
      run_cypher("MATCH (:ExPerson)-[r:ADOPTED_CHILD_OF]->(:ExPerson) "
                 "RETURN count(r) AS n")[0]['n'], '개')

In [ ]:
# '성종의 아버지는?' 타입을 나눠 뒀으니 어느 쪽을 묻는지가 쿼리에 드러난다
print('혈통으로:', run_cypher("MATCH (:ExPerson {name: '성종'})-[:BIOLOGICAL_CHILD_OF]->(p:ExPerson) "
                            "RETURN p.name AS 아버지"))
# 왕위 계승에서 성종의 아버지는 숙부인 예종이다. 즉위의 근거가 된 관계다
print('법통으로:', run_cypher("MATCH (:ExPerson {name: '성종'})-[:ADOPTED_CHILD_OF]->(p:ExPerson) "
                            "RETURN p.name AS 아버지"))

In [ ]:
# '성종의 윗대를 모두 대라' 를 두 번 묻는다. 따라가는 관계 타입만 바꾼다
# 세로줄(|) 은 '이 타입들 중 아무거나' 라는 뜻이다. 두 경로가 겹쳐 같은 사람이 두 번 나오므로 DISTINCT 를 쓴다
bio = run_cypher("MATCH (:ExPerson {name: '성종'})-[:BIOLOGICAL_CHILD_OF*]->(p:ExPerson) "
                 "RETURN DISTINCT p.name AS name ORDER BY name")
both = run_cypher("MATCH (:ExPerson {name: '성종'})"
                  "-[:BIOLOGICAL_CHILD_OF|ADOPTED_CHILD_OF*]->(p:ExPerson) "
                  "RETURN DISTINCT p.name AS name ORDER BY name")
print('혈통만  :', [r['name'] for r in bio])
print('입적 포함:', [r['name'] for r in both])

> **같은 질문에 다른 답입니다.** 입적을 포함하면 **예종**이 더해집니다. 예종은 성종의 숙부이지 혈통상 윗대가 아닙니다. 어느 쪽이 맞느냐는 데이터가 정하지 않습니다. **질문이 정합니다.** 족보를 그리는 일이라면 혈통만, 왕위 계승을 따지는 일이라면 입적까지 따라갑니다.

그렇다면 타입을 하나로 합치고 속성으로 구분하면 안 될까요? `(:ExPerson)-[:CHILD_OF {kind: '생부'}]->` 처럼요. 되긴 됩니다. 나란히 만들어 견줘 봅시다.

In [ ]:
# 같은 사실을 타입 하나에 몰고 kind 속성으로 구분한 안을 나란히 만든다
run_cypher("""
MATCH (sk:ExPerson {name: '성종'}), (uk:ExPerson {name: '의경세자'}), (yj:ExPerson {name: '예종'}),
      (sjo:ExPerson {name: '세조'}), (sj:ExPerson {name: '세종'}), (tj:ExPerson {name: '태종'})
MERGE (sk)-[:CHILD_OF {kind: '생부'}]->(uk)
MERGE (sk)-[:CHILD_OF {kind: '양부'}]->(yj)
MERGE (uk)-[:CHILD_OF {kind: '생부'}]->(sjo)
MERGE (yj)-[:CHILD_OF {kind: '생부'}]->(sjo)
MERGE (sjo)-[:CHILD_OF {kind: '생부'}]->(sj)
MERGE (sj)-[:CHILD_OF {kind: '생부'}]->(tj)
""")
print('합친 안의 관계:', run_cypher("MATCH (:ExPerson)-[r:CHILD_OF]->(:ExPerson) "
                                "RETURN count(r) AS n")[0]['n'], '개')

In [ ]:
# 합친 안에서 '혈통만' 을 물으려면 경로에 놓인 관계를 하나씩 뜯어 조건을 걸어야 한다
# relationships(p) 는 경로가 지나온 관계들이고, all(...) 은 그 전부가 조건을 만족하는지 본다
guarded = run_cypher("MATCH p = (:ExPerson {name: '성종'})-[:CHILD_OF*]->(q:ExPerson) "
                     "WHERE all(r IN relationships(p) WHERE r.kind = '생부') "
                     "RETURN DISTINCT q.name AS name ORDER BY name")
# 그 조건을 빠뜨리면 양부 쪽 경로가 조용히 섞여 든다. 에러는 나지 않는다
unguarded = run_cypher("MATCH p = (:ExPerson {name: '성종'})-[:CHILD_OF*]->(q:ExPerson) "
                       "RETURN DISTINCT q.name AS name ORDER BY name")
print('조건 건 것 :', [r['name'] for r in guarded])
print('조건 빠뜨림:', [r['name'] for r in unguarded])

> 조건을 건 결과는 타입을 나눈 안과 **같습니다.** 다른 것은 **틀리기 쉬운 정도**입니다. 타입을 나누면 `-[:BIOLOGICAL_CHILD_OF*]->` 라고 쓰는 순간 혈통만 따라갑니다. 합쳐 두면 `all(r IN relationships(p) WHERE r.kind = '생부')` 를 **매번** 붙여야 하고, 한 번 빠뜨리면 답이 조용히 늘어납니다. 에러가 아니라 **틀린 답**입니다.

게다가 관계 타입은 저장 구조의 일부라 `-[:BIOLOGICAL_CHILD_OF]->` 는 그 타입의 관계만 훑습니다. 합친 안은 전부 훑은 뒤 속성으로 거릅니다.

**기준**: 뜻이 다르고 그것만 따로 찾는 일이 잦으면 **타입을 나눕니다**. 같은 사건인데 값만 다르면 **관계 속성**으로 둡니다.

## 5-2. 방향은 한 번만 정합니다

관계에는 방향이 있습니다. **저장할 때는 방향이 반드시 정해지지만, 찾을 때는 화살표를 빼면** 양쪽 다 걸립니다. 그래서 방향은 **뜻이 자연스러운 쪽**으로 한 번만 정하면 됩니다.

문제는 **대칭 관계**입니다. 문종과 세조는 둘 다 세종의 아들이라 형제입니다. "문종이 세조의 형이면 세조는 문종의 동생"이니 양쪽으로 두 번 저장하고 싶어집니다. 그러면 안 됩니다.

In [ ]:
# 문종과 세조는 둘 다 세종의 아들이다. 누가 형인지는 태어난 해가 말해 준다
run_cypher("""
MERGE (sj:ExPerson {name: '세종'})
MERGE (mj:ExPerson {name: '문종'})  SET mj.born = 1414
MERGE (sjo:ExPerson {name: '세조'}) SET sjo.born = 1417
MERGE (mj)-[:BIOLOGICAL_CHILD_OF]->(sj)
MERGE (mj)-[:SIBLING_OF]->(sjo)
""")
for row in run_cypher("MATCH (p:ExPerson) WHERE p.born IS NOT NULL "
                      "RETURN p.name AS 이름, p.born AS 태어난해 ORDER BY 태어난해"):
    print(row)

In [ ]:
# 형제 관계는 한 방향으로만 저장했다. 화살표를 붙여 세면 저장한 쪽에서만 걸린다
print('문종에서 ->:', run_cypher("MATCH (:ExPerson {name: '문종'})-[:SIBLING_OF]->(x) "
                               "RETURN count(*) AS n")[0]['n'])
print('세조에서 ->:', run_cypher("MATCH (:ExPerson {name: '세조'})-[:SIBLING_OF]->(x) "
                               "RETURN count(*) AS n")[0]['n'])
print('화살표 없이:', run_cypher("MATCH (:ExPerson)-[:SIBLING_OF]-(x) "
                               "RETURN count(*) AS n")[0]['n'])

> 저장한 관계는 **하나**입니다. 문종에서 화살표를 따라가면 1건, 세조에서 따라가면 0건, 화살표를 빼면 2건입니다. 마지막 2건은 관계가 둘이라는 뜻이 아니라 **같은 관계를 양쪽 끝에서 한 번씩 봤다**는 뜻입니다. 그래서 쌍을 셀 때는 `WHERE a.name < b.name` 처럼 순서를 주어 한 번만 세게 합니다.

> 대칭 관계를 **양쪽으로 두 번 저장하면** 관계 수가 두 배가 되고, 하나만 지우면 그래프가 어긋납니다. **저장은 한 방향, 조회는 화살표를 빼는 것**이 정석입니다.

## 5-3. 연결 자체의 성질은 관계 속성으로

값이 **한쪽 노드**를 설명하면 그 노드의 속성입니다. **두 노드 사이**를 설명하면 관계의 속성입니다.

- 정조가 태어난 해는 정조의 속성입니다.
- **정조가 효장세자의 양자로 입적된 것을 누가 정했는가**는 정조의 값도 효장세자의 값도 아닙니다. 그 **연결**의 성질입니다. 영조의 명이었죠.

In [ ]:
# 입적을 정한 사람은 영조다. 두 사람 중 누구의 속성도 아니라서 관계에 붙인다
run_cypher("""
MATCH (jj:ExPerson {name: '정조'})-[r:ADOPTED_CHILD_OF]->(:ExPerson {name: '효장세자'})
SET r.decided_by = '영조'
""")
print(run_cypher("MATCH (a:ExPerson)-[r:ADOPTED_CHILD_OF]->(b:ExPerson) "
                 "WHERE r.decided_by IS NOT NULL "
                 "RETURN a.name AS 입적한사람, b.name AS 양부, r.decided_by AS 정한사람"))

In [ ]:
# 왕위를 물려준 관계에는 '언제' 가 붙는다. 대수가 바로 이어지는 다섯 자리만 잇는다
# UNWIND 는 리스트를 한 줄씩 풀어 준다. 같은 모양의 관계 여럿을 한 번에 만들 때 쓴다
run_cypher("""
UNWIND [['태종', '세종', 1418], ['세종', '문종', 1450], ['세조', '예종', 1468],
        ['예종', '성종', 1469], ['영조', '정조', 1776]] AS row
MATCH (a:ExPerson {name: row[0]}), (b:ExPerson {name: row[1]})
MERGE (a)-[r:SUCCEEDED_BY]->(b)
SET r.year = row[2]
""")
for row in run_cypher("MATCH (a:ExPerson)-[r:SUCCEEDED_BY]->(b:ExPerson) "
                      "RETURN a.name AS 물려준사람, b.name AS 물려받은사람, r.year AS 해 "
                      "ORDER BY 해"):
    print(row)

> 즉위한 해는 **물려준 사람의 값도 물려받은 사람의 값도 아닙니다.** 그 계승이라는 사건의 값입니다. 그래서 관계에 붙였습니다.

## 관계 타입 이름을 짓는 법

| 나쁜 이름 | 왜 나쁜가 | 대신 |
|---|---|---|
| `:HAS` | 무엇을 가졌다는 것인지 말하지 않는다. 온갖 뜻이 한 타입에 쌓인다 | `:REIGNED`, `:HAPPENED_IN` |
| `:RELATED_TO` | 관계가 있다는 말만 한다. 그래프에 있는 것은 전부 관계다 | `:BIOLOGICAL_CHILD_OF` |
| `:LINK`, `:CONNECTED` | 같다. 이름이 아무것도 말하지 않는다 | 동사구로 짓는다 |
| `:PARENT`(명사) | 어느 쪽이 부모인지 방향과 함께 읽히지 않는다 | `:CHILD_OF` 또는 `:PARENT_OF` |

> 좋은 관계 타입 이름은 **읽으면 문장이 됩니다.** `(정조)-[:BIOLOGICAL_CHILD_OF]->(사도세자)` 는 "정조는 사도세자의 혈통상 자식이다"로 읽힙니다.

## 판단 기준: 타입·속성·방향

| 이런 경우 | 어떻게 |
|---|---|
| 뜻이 **다른 사건**이고 그것만 따로 찾는 일이 잦다 | **타입을 나눈다** |
| 양 끝 노드의 **종류 조합이 다르다** | **타입을 나눈다** |
| 같은 사건인데 **값만 다르다** | **관계 속성**으로 둔다 |
| 방향이 **뜻을 가른다** | 자연스러운 쪽으로 한 번만 저장한다 |
| 관계가 **대칭이다** | 한 방향만 저장하고, 조회에서 화살표를 뺀다 |
| **셋 이상이 얽힌다** | 그 사실 자체를 노드로 승격한다(7절) |

**그럼 의료 데이터에서는?** 다음 교안의 의료 그래프가 `TREATS`(병 자체를 조절한다)와 `PALLIATES`(증상을 눅인다)를 **따로 둔 것**이 5-1 의 결정입니다. 하나로 합쳐 두면 조건을 빠뜨린 쿼리가 "증상만 눅이는 약"을 "치료하는 약"으로 내놓습니다. 의료에서 그건 버그가 아니라 **틀린 사실**입니다. `UPREGULATES_CG` 와 `UPREGULATES_DG` 는 양 끝의 종류가 달라서 나뉜 경우이고, 그 덕에 타입 이름만 보고 양 끝이 무엇인지 압니다.

### ✅ 바로 확인 퀴즈

**1.** `BIOLOGICAL_CHILD_OF` 와 `ADOPTED_CHILD_OF` 를 `CHILD_OF` 하나로 합치고 `kind` 속성으로 구분하면 어떤 사고가 생길 수 있나요?

<details><summary>정답 보기</summary>

가변 길이 경로에서 `all(r IN relationships(p) WHERE r.kind = '생부')` 조건을 **빠뜨리면** 양부 쪽 경로가 섞여 들어옵니다. 에러가 아니라 **답이 조용히 늘어납니다.** 타입을 나눠 두면 `-[:BIOLOGICAL_CHILD_OF*]->` 라고 쓰는 순간 그런 실수가 불가능해집니다.

</details>

**2.** 대칭 관계(형제)를 양쪽 방향으로 두 번 저장하면 무엇이 나빠지나요?

<details><summary>정답 보기</summary>

관계 수가 **두 배**가 되고, 두 관계 중 하나만 지우거나 고치면 그래프가 **어긋납니다.** 한 방향만 저장하고 조회에서 화살표를 빼면 저장은 절반이고 결과는 같습니다. 다만 화살표를 뺀 매칭은 같은 관계를 양쪽 끝에서 한 번씩 보므로, 쌍을 셀 때는 이름에 순서를 주어야 합니다.

</details>

**3.** 왕위를 물려준 해(`year`)를 물려받은 사람의 속성으로 두면 무엇이 아쉬운가요?

<details><summary>정답 보기</summary>

그 해는 한 사람의 값이 아니라 **계승이라는 연결의 값**입니다. 사람의 속성으로 두면 "누구에게서 물려받았는가"와 떨어져, 물려준 사람이 누구인지 그 값만 보고는 알 수 없습니다. 두 노드 사이를 설명하는 값은 관계에 붙입니다.

</details>

In [ ]:
# 5절 실험 노드를 지운다. DETACH 가 붙은 관계까지 함께 지워 준다
run_cypher("MATCH (n:ExPerson) DETACH DELETE n")
print('남은 5절 노드:', run_cypher("MATCH (n:ExPerson) RETURN count(n) AS n")[0]['n'], '개')

---
# 6. 라벨로 둘까요, 속성으로 둘까요?

사건을 담다 보면 "이건 국내에서 일어난 일, 저건 세계에서 일어난 일" 같은 구분이 생깁니다. 이 구분을 **라벨**(`:ExEvent:ExDomestic`)로 둘 수도 있고 **속성**(`scope: '국내'`)으로 둘 수도 있습니다. 3절의 "노드냐 속성이냐"와 닮았지만 다른 결정입니다. 여기서는 **노드 하나를 어떻게 분류할 것인가**를 정합니다.

<img src="images/label_or_property.png" width="880">

> 그림은 두 안을 같은 `:ExEvent` 라벨로 그렸습니다. 아래 코드는 **두 안을 한 데이터베이스에 나란히 놓고 견주려고** 속성 안만 `:ExScopeEvent` 로 이름을 갈라 뒀습니다. 판단이 달라지지는 않습니다.

두 안을 나란히 적재해 봅시다.

In [ ]:
# 라벨 안: 구분을 라벨로 붙인다. 한 노드가 라벨을 여럿 가질 수 있다
run_cypher("""
MERGE (a:ExEvent:ExDomestic {name: '대한제국 선포'}) SET a.year = 1897
MERGE (b:ExEvent:ExDomestic {name: '강화도조약'})    SET b.year = 1876
MERGE (e:ExEvent:ExDomestic {name: '임진왜란'})      SET e.year = 1592
MERGE (c:ExEvent:ExWorld {name: '프랑스 혁명'})      SET c.year = 1789
MERGE (d:ExEvent:ExWorld {name: '메이지 유신'})      SET d.year = 1868
""")
print('라벨 안 국내:', run_cypher("MATCH (e:ExEvent:ExDomestic) RETURN count(e) AS n")[0]['n'],
      '· 세계:', run_cypher("MATCH (e:ExEvent:ExWorld) RETURN count(e) AS n")[0]['n'])

In [ ]:
# 속성 안: 같은 구분을 scope 라는 칸 하나에 담는다
run_cypher("""
MERGE (a:ExScopeEvent {name: '대한제국 선포'}) SET a.year = 1897, a.scope = '국내'
MERGE (b:ExScopeEvent {name: '강화도조약'})    SET b.year = 1876, b.scope = '국내'
MERGE (e:ExScopeEvent {name: '임진왜란'})      SET e.year = 1592, e.scope = '국내'
MERGE (c:ExScopeEvent {name: '프랑스 혁명'})   SET c.year = 1789, c.scope = '세계'
MERGE (d:ExScopeEvent {name: '메이지 유신'})   SET d.year = 1868, d.scope = '세계'
""")
print('속성 안 국내:', run_cypher("MATCH (e:ExScopeEvent) WHERE e.scope = '국내' "
                               "RETURN count(e) AS n")[0]['n'],
      '· 세계:', run_cypher("MATCH (e:ExScopeEvent) WHERE e.scope = '세계' "
                          "RETURN count(e) AS n")[0]['n'])

In [ ]:
# 질문) '세계 쪽 사건만 대라'. 라벨 안은 라벨이 곧 조건이라 훑는 범위가 그 라벨로 좁혀진다
print('라벨 안:', run_cypher("MATCH (e:ExEvent:ExWorld) "
                           "WITH e ORDER BY e.year RETURN collect(e.name) AS 사건"))
# 속성 안은 그 라벨을 다 훑은 뒤 값을 견줘 거른다. 답은 같다
print('속성 안:', run_cypher("MATCH (e:ExScopeEvent) WHERE e.scope = '세계' "
                           "WITH e ORDER BY e.year RETURN collect(e.name) AS 사건"))

> 답은 같습니다. 차이는 **분류가 바뀔 때** 드러납니다. 국내와 세계 둘로는 담기지 않는 사건이 있습니다. 강화도조약은 조선과 일본 사이의 일이니 "대외"라는 갈래를 새로 두기로 했다고 해 봅시다. 두 안에서 그 일을 각각 해 봅니다.

In [ ]:
# 갈래를 하나 더 두기로 했다. 라벨 안은 떼고 붙이는 두 동작이고, 속성 안은 값 하나를 덮어쓰면 된다
run_cypher("MATCH (e:ExEvent {name: '강화도조약'}) REMOVE e:ExDomestic SET e:ExForeign")
run_cypher("MATCH (e:ExScopeEvent {name: '강화도조약'}) SET e.scope = '대외'")
print('라벨 안 국내:', run_cypher("MATCH (e:ExEvent:ExDomestic) RETURN count(e) AS n")[0]['n'],
      '· 대외:', run_cypher("MATCH (e:ExEvent:ExForeign) RETURN count(e) AS n")[0]['n'])
# 속성 안은 값만 바뀌었을 뿐 그래프의 모양이 그대로다. 갈래가 늘어도 스키마는 늘지 않는다
print('속성 안 국내:', run_cypher("MATCH (e:ExScopeEvent) WHERE e.scope = '국내' "
                               "RETURN count(e) AS n")[0]['n'],
      '· 대외:', run_cypher("MATCH (e:ExScopeEvent) WHERE e.scope = '대외' "
                          "RETURN count(e) AS n")[0]['n'])

> 두 안 모두 국내가 둘, 대외가 하나가 됐습니다. 다만 라벨 안에서는 **`ExForeign` 이라는 라벨이 새로 생겼습니다.** 갈래가 늘 때마다 그래프의 **모양 자체**가 늘어납니다. 속성 안에서는 값 하나가 바뀌었을 뿐입니다.

## 판단 기준: 라벨일까요, 속성일까요?

| 그 값이 | 어디에 | 왜 |
|---|---|---|
| 가짓수가 **적고**(대략 열 이하) 거의 **바뀌지 않는다** | **라벨** | 라벨은 훑는 단위이고, 인덱스가 라벨마다 걸린다 |
| 그 값으로 잘라 보는 일이 **아주 잦다** | **라벨** | 패턴에 바로 써서 조건 없이 걸러진다 |
| 가짓수가 많거나 **자주 바뀐다** | **속성** | 라벨이 늘면 스키마가 늘고, 옮기려면 떼고 붙여야 한다 |
| **크기를 견주거나 범위로 자른다** | **속성** | 라벨에는 순서가 없다 |
| **모른다**를 표현해야 한다 | **속성** | 라벨은 붙거나 안 붙거나 둘뿐이다 |

> **라벨을 값처럼 쓰지 마세요.** 노드 하나에 라벨이 셋을 넘어가면 대개 속성으로 둘 것을 라벨로 두고 있는 것입니다. 라벨은 "이 노드가 **무엇인가**"이지 "이 노드의 **값이 무엇인가**"가 아닙니다. `:Event:Domestic:Treaty:NineteenthCentury:Important` 같은 노드를 보면 뒤의 셋은 속성이어야 합니다.

**그럼 의료 데이터에서는?** 의료 그래프는 라벨 5종을 오직 **노드의 종류**로만 씁니다(약물·질병·유전자·증상·약효분류). 다섯 개뿐이고 바뀌지 않으니 라벨이 맞습니다. 반대로 "승인된 약인가" 같은 구분은 시간에 따라 바뀌는 값이라 라벨이 아니라 **속성**이 맞습니다.

### ✅ 바로 확인 퀴즈

**1.** 사건의 연도(`1897`)를 라벨(`:Year1897`)로 두면 무엇이 나쁜가요?

<details><summary>정답 보기</summary>

**가짓수가 너무 많고 순서를 쓸 수 없습니다.** 연도마다 라벨이 하나씩 생겨 스키마가 수천 개로 늘어나고, "1800년 이후" 같은 범위 조건을 걸 수 없습니다. 라벨에는 크기 비교가 없습니다. 연도는 속성으로 두고, 묶어 볼 일이 잦으면 **연도 노드**로 올립니다(4절).

</details>

**2.** 반대로 노드의 종류(사람인가 사건인가)를 라벨이 아니라 속성(`type: 'person'`)으로 두면 무엇이 아쉬운가요?

<details><summary>정답 보기</summary>

**패턴에 쓸 수 없습니다.** `MATCH (p:ExPerson)-[:CHILD_OF]->(q:ExPerson)` 처럼 양 끝의 종류를 패턴에 적으면 그 종류만 훑는데, 속성으로 두면 전부 훑은 뒤 걸러야 합니다. 종류는 가짓수가 적고 바뀌지 않으니 라벨이 맞습니다.

</details>

**3.** 한 노드에 라벨을 다섯 개 붙였다면 무엇을 의심해야 하나요?

<details><summary>정답 보기</summary>

**속성으로 둘 것을 라벨로 두고 있는지** 의심합니다. 라벨은 "이 노드가 무엇인가"를 말하는 자리라 보통 하나면 충분하고, 분류가 겹칠 때만 둘이나 셋이 됩니다. 그보다 늘어나면 값을 라벨로 표현하고 있는 것이고, 그 값이 바뀔 때마다 라벨을 떼고 붙여야 합니다.

</details>

In [ ]:
# 6절 실험 노드를 지운다. ExDomestic 같은 추가 라벨이 붙은 노드도 ExEvent 로 함께 잡힌다
run_cypher("MATCH (n) WHERE n:ExEvent OR n:ExScopeEvent DETACH DELETE n")
print('남은 6절 노드:', run_cypher("MATCH (n) WHERE n:ExEvent OR n:ExScopeEvent "
                                "RETURN count(n) AS n")[0]['n'], '개')

---
# 7. 중간 노드: 셋 이상이 얽힐 때

관계는 **두 노드 사이**를 잇습니다. 그래서 둘짜리 사실은 관계로 잘 담깁니다. 문제는 셋 이상이 얽힐 때입니다.

> "세종은 **4대** 임금으로 **1418년부터 1450년까지** 왕위에 있었고, **그동안 훈민정음이 창제됐다.**"

여기에는 임금·대수·기간·사건이 함께 들어 있습니다. 3절에서 재위를 노드로 뺐던 이유가 여기서 분명해집니다. 먼저 관계 하나에 담아 보고 **어디서 막히는지** 봅시다.

In [ ]:
# 재위를 관계에 담아 본다. 대수와 기간을 관계 속성으로 붙이면 여기까지는 된다
run_cypher("""
MERGE (t:ExThrone {name: '조선'})
MERGE (k:ExPerson {name: '세종'})
MERGE (k)-[r:REIGNED_IN]->(t)
SET r.order_no = 4, r.start_year = 1418, r.end_year = 1450
""")
print(run_cypher("MATCH (:ExPerson)-[r:REIGNED_IN]->(:ExThrone) "
                 "RETURN r.order_no AS 대수, r.start_year AS 시작, r.end_year AS 끝"))

In [ ]:
# 이제 '훈민정음 창제는 그 재위 중에 일어났다' 를 담아 보자. 그 재위는 관계라서 화살표를 붙일 수 없다
# 아래는 일부러 실패시키는 시연이다. 에러 이름을 확인하고 넘어간다
try:
    run_cypher("""
    MATCH (:ExPerson {name: '세종'})-[r:REIGNED_IN]->(:ExThrone)
    MERGE (e:ExEvent {name: '훈민정음 창제'})-[:DURING]->(r)
    """)
except Exception as error:
    print('관계에는 관계를 붙일 수 없습니다:', type(error).__name__)

> 문법 오류입니다. Cypher 는 화살표의 끝에 **노드**가 오기를 요구합니다. 관계는 두 노드를 잇는 선일 뿐이라 **그 자체를 가리킬 수 없습니다.** 재위에 사건을 이으려면 재위가 **노드**여야 합니다.

이렇게 관계로는 담기지 않는 사실을 노드로 올리는 것을 **중간 노드로 승격한다**고 말합니다.

<img src="images/intermediate_node.png" width="880">

In [ ]:
# 재위 자체를 노드로 올린다. 그러면 임금도 사건도 그 노드에 이어 붙일 수 있다
run_cypher("""
MERGE (k:ExPerson {name: '세종'})
MERGE (t:ExThrone {name: '조선'})
MERGE (r:ExReign {order_no: 4}) SET r.start_year = 1418, r.end_year = 1450
MERGE (k)-[:REIGNED]->(r)
MERGE (r)-[:ON_THRONE]->(t)
MERGE (e:ExEvent {name: '훈민정음 창제'}) SET e.lunar_year = 1443
MERGE (e)-[:DURING]->(r)
""")
print('재위 노드에 붙은 관계:', run_cypher("MATCH (:ExReign {order_no: 4})-[]-(x) "
                                      "RETURN count(*) AS n")[0]['n'], '개')

In [ ]:
# 이제 '훈민정음 창제는 몇 대 임금 때인가' 를 사건에서 재위를 거쳐 임금까지 2홉으로 묻는다
print(run_cypher("MATCH (e:ExEvent {name: '훈민정음 창제'})-[:DURING]->(r:ExReign)"
                 "<-[:REIGNED]-(k:ExPerson) "
                 "RETURN k.name AS 임금, r.order_no AS 대수, e.lunar_year AS 창제연도"))

> 재위 노드에 관계가 셋 붙었습니다. 임금 쪽으로 하나, 왕위 쪽으로 하나, 사건 쪽으로 하나입니다. 그 노드를 징검다리 삼아 사건에서 임금까지 건너갔습니다. 재위를 임금의 속성 칸으로 뒀다면(3절 안 A) 이 질문은 담을 자리가 없어 답할 수 없었습니다.

## 언제 승격하나요?

| 이런 사실이면 | 어떻게 |
|---|---|
| 두 개체 **사이의 성질**이다 | 관계 속성으로 충분하다 |
| **셋 이상**이 함께 얽힌다 | 중간 노드로 올린다 |
| 그 사실에 **다른 것을 이어 붙일** 일이 있다 | 중간 노드로 올린다 |
| 같은 두 개체 사이에 **여러 건**이 쌓이고 시점이 붙는다 | 중간 노드로 올린다 |
| 그 사실 **자체를 세거나 정렬**한다 | 중간 노드로 올린다 |

> 중간 노드는 관계형에서 **연결 표**를 만드는 것과 같은 결정입니다. 다른 점은 그래프에서 그 노드가 **경로의 징검다리**가 된다는 것입니다. 연결 표는 조인의 대상일 뿐이지만, 중간 노드는 그 자체가 출발점도 도착점도 될 수 있습니다.

> 다만 **아무 관계나 승격하지는 않습니다.** 노드가 하나 늘 때마다 관계가 둘 늘고, 1홉이던 질문이 2홉이 됩니다. 셋 이상이 얽히거나 이어 붙일 것이 있을 때만 올립니다.

**그럼 의료 데이터에서는?** 처방이나 투여는 **환자·약·시점·용량**이 함께 얽힌 사실이라 중간 노드가 됩니다. 다음 교안이 쓰는 자료는 문헌에 정리된 관계만 담고 있어 중간 노드가 없지만, 실제 병원 자료를 그래프로 옮기면 여기가 첫 설계 자리입니다.

### ✅ 바로 확인 퀴즈

**1.** 재위를 `(:ExPerson)-[:REIGNED_IN {order_no, start_year, end_year}]->(:ExThrone)` 관계로 두면 무엇을 할 수 없나요?

<details><summary>정답 보기</summary>

**그 재위에 다른 것을 이어 붙일 수 없습니다.** 관계는 두 노드를 잇는 선이라 화살표의 끝이 될 수 없습니다. "그 재위 중에 일어난 사건"을 담으려면 재위가 노드여야 합니다.

</details>

**2.** 그렇다고 모든 관계를 노드로 올리면 안 되는 이유는?

<details><summary>정답 보기</summary>

노드가 하나 늘 때마다 **관계가 둘 늘고**, 1홉이던 질문이 2홉이 됩니다. 저장 공간과 적재 시간이 들고 쿼리가 길어집니다. 셋 이상이 얽히거나 그 사실에 이어 붙일 것이 있을 때만 올립니다.

</details>

**3.** "같은 두 사람 사이에 여러 번 일어난 일"은 왜 중간 노드가 좋을까요?

<details><summary>정답 보기</summary>

관계 속성으로 두면 한 관계에 한 벌만 담을 수 있어, 여러 건을 담으려면 관계를 여러 개 만들고 그 각각을 구분할 방법이 따로 필요합니다. 사건 자체를 노드로 올리면 **한 건이 노드 하나**가 되어 세고 정렬하고 다른 것에 잇기가 모두 쉬워집니다.

</details>

In [ ]:
# 7절 실험 노드를 지운다
run_cypher("MATCH (n) WHERE n:ExPerson OR n:ExThrone OR n:ExReign OR n:ExEvent DETACH DELETE n")
print('남은 7절 노드:', run_cypher("MATCH (n) WHERE n:ExPerson OR n:ExThrone OR n:ExReign "
                                "OR n:ExEvent RETURN count(n) AS n")[0]['n'], '개')

---
# 8. 무엇을 키로 삼을까요?

`MERGE (n:ExKing {...})` 의 중괄호 안은 **"이미 있는지 찾을 때 쓰는 조건"** 입니다. 곧 **무엇이 같은 것인가**를 정하는 자리입니다.

조선 임금은 이름이 여럿입니다. **묘호**(세종)는 죽은 뒤 종묘에 올리며 붙인 이름이고, **휘**(이도)는 본인의 이름입니다. 세종과 이도는 **같은 사람**인데 부르는 이름이 다릅니다. 태조는 기록에 **이단**과 **이성계** 두 이름이 남아 있습니다.

자료 두 벌이 각각 다른 이름을 쓰고 있다면 어떻게 될까요?

In [ ]:
# 이름을 키로 삼는다. 묘호로 적힌 자료와 휘로 적힌 자료가 각각 들어온다고 하자
run_cypher("""
MERGE (a:ExNameKey {name: '세종'}) SET a.source = '묘호로 적힌 자료'
MERGE (b:ExNameKey {name: '이도'}) SET b.source = '휘로 적힌 자료'
""")
print('이름을 키로:', run_cypher("MATCH (n:ExNameKey) RETURN count(n) AS n")[0]['n'], '개')

In [ ]:
# 대수를 키로 삼는다. 이름은 표시용 속성으로 내린다
run_cypher("""
MERGE (a:ExIdKey {order_no: 4}) SET a.temple_name = '세종'
MERGE (b:ExIdKey {order_no: 4}) SET b.given_name = '이도'
""")
print('대수를 키로:', run_cypher("MATCH (n:ExIdKey) RETURN count(n) AS n")[0]['n'], '개')

In [ ]:
# 두 줄이 한 노드로 모였고 묘호와 휘가 그 노드의 두 속성이 됐다
print(run_cypher("MATCH (n:ExIdKey) "
                 "RETURN n.order_no AS 대수, n.temple_name AS 묘호, n.given_name AS 휘"))

> 이름을 키로 쓰면 같은 사람이 **두 노드**가 됩니다. 에러도 경고도 없습니다. 대수를 키로 쓰면 두 줄이 **한 노드**로 모이고 묘호와 휘가 그 노드의 속성이 됩니다.

> **그런데 대수도 만능이 아닙니다.** 대수가 겹치지 않는 것은 **조선 안에서**입니다. 다른 나라나 다른 왕조의 임금까지 담으면 4대가 여럿이 됩니다. 키는 **어느 범위 안에서 유일한가**를 함께 정해야 하고, 자료의 범위를 넓히는 순간 키를 다시 골라야 합니다.

> **"이름이 같으면 같은 것인가"는 데이터가 답해 주지 않습니다. 그 분야를 아는 사람이 답합니다.** 모델링에서 가장 자주 틀리는 자리입니다. 지금 안전한 키가 다음 분기에도 안전하다는 보장은 없습니다.

## 정한 키를 제약으로 잠급니다

키를 정했으면 **데이터베이스에도 알려 줍니다.** 문법과 종류는 **교안_01 1절**에서 이미 다뤘습니다(`CREATE CONSTRAINT ... REQUIRE ... IS NODE KEY`, 그리고 `UNIQUE` 는 유일성만·`NODE KEY` 는 유일성과 존재를 함께 요구한다는 것). 여기서는 그 차이가 **실제로 무엇을 통과시키고 무엇을 막는지** 실행으로 봅니다.

> `IS NODE KEY` 는 **Enterprise 기능**입니다. Neo4j Desktop 에는 Enterprise 개발자 라이선스가 딸려 오지만, Docker·Homebrew 의 기본 이미지는 커뮤니티라 이 문법이 막힙니다. 그때는 `IS UNIQUE` 로 바꿔 커뮤니티라면 아래 두 번째 셀이 에러로 멈춥니다. 그 에러 자체가 이 문법이 Enterprise 기능이라는 증거이고, 이 절이 보이려는 대비는 그때 확인할 수 없습니다.

In [ ]:
# UNIQUE 를 걸어 두고, 키가 아예 없는 노드를 두 개 넣어 본다
run_cypher("CREATE CONSTRAINT ex_unique_demo IF NOT EXISTS "
           "FOR (n:ExUniqueDemo) REQUIRE n.order_no IS UNIQUE")
run_cypher("CREATE (:ExUniqueDemo {temple_name: '세종'})")
run_cypher("CREATE (:ExUniqueDemo {temple_name: '문종'})")
# null 끼리는 '같다' 고 보지 않아서 유일성 제약을 그냥 통과한다
print('키가 빈 노드:', run_cypher("MATCH (n:ExUniqueDemo) WHERE n.order_no IS NULL "
                               "RETURN count(n) AS n")[0]['n'], '개')

In [ ]:
# 같은 자리에 NODE KEY 를 걸면 어떻게 되는지 본다
run_cypher("CREATE CONSTRAINT ex_nodekey_demo IF NOT EXISTS "
           "FOR (n:ExNodeKeyDemo) REQUIRE n.order_no IS NODE KEY")
try:
    run_cypher("CREATE (:ExNodeKeyDemo {temple_name: '세종'})")   # 키를 빠뜨렸다
except Exception as error:
    print('NODE KEY 는 막습니다:', type(error).__name__)
# 키를 채우면 들어간다. 막는 것은 '키가 빈 노드' 뿐이다
run_cypher("CREATE (:ExNodeKeyDemo {order_no: 4, temple_name: '세종'})")
print('들어간 노드:', run_cypher("MATCH (n:ExNodeKeyDemo) RETURN count(n) AS n")[0]['n'], '개')

> `UNIQUE` 는 **값이 있는 것들끼리만** 겹치지 않게 합니다. `null` 은 서로 같다고 보지 않아서 키가 빈 노드는 두 개가 그냥 들어왔습니다. 적재는 성공했다고 나오는데 **다시 찾을 수 없는 노드**가 쌓입니다.

> `NODE KEY` 는 그 구멍을 막습니다. 유일성에 **존재**까지 얹으니, "이것이 키다"라는 설계 결정을 데이터베이스가 대신 지켜 줍니다. 키를 빠뜨린 쪽은 에러로 멈췄고, 채운 쪽은 들어갔습니다.

> 다만 제약은 **라벨마다** 걸립니다. `:ExKing` 에 건 것이 `:ExEvent` 를 지켜 주지 않습니다. 라벨이 5종이면 5개를 겁니다.

**그럼 의료 데이터에서는?** 다음 교안이 이름 대신 `id`(`Compound::DB00396` 같은 값)를 키로 쓰는 것이 이 결정입니다. 그 자료에는 **이름이 같은데 종류가 다른** 항목이 있어서, 이름으로 합치면 노드가 조용히 줄어듭니다. 그리고 그 `id` 에 `IS NODE KEY` 를 걸어 두면 키가 빈 행이 섞여 들어올 때 적재가 **에러로 멈춥니다.**

### ✅ 바로 확인 퀴즈

**1.** 이름을 키로 `MERGE` 해서 같은 사람이 두 노드가 되면 어떤 신호가 뜨나요?

<details><summary>정답 보기</summary>

**아무 신호도 뜨지 않습니다.** 에러도 경고도 없고 적재는 성공으로 끝납니다. 나중에 **적재된 노드 수를 원본 건수와 대조**해야 비로소 드러납니다. 그래서 적재는 항상 건수로 검증합니다.

</details>

**2.** `MERGE (n:ExKing {order_no: 4}) SET n.temple_name = '세종'` 에서 이름을 `MERGE` 의 중괄호가 아니라 `SET` 으로 채우는 이유는?

<details><summary>정답 보기</summary>

`MERGE` 의 중괄호는 **찾을 때 쓰는 식별 조건**입니다. 이름까지 넣으면 표기가 조금만 달라져도 다른 노드로 취급돼 중복이 생깁니다. **변하지 않는 식별자만** `MERGE` 에 두고 나머지는 `SET` 합니다.

</details>

**3.** `IS UNIQUE` 를 걸어 두었는데도 키가 빈 노드가 쌓이는 이유는?

<details><summary>정답 보기</summary>

`null` 끼리는 **같다고 보지 않기** 때문입니다. 유일성 제약은 "값이 있는 것들끼리 겹치지 마라"이지 "값이 있어야 한다"가 아닙니다. 키가 비는 것까지 막으려면 `IS NODE KEY` 를 씁니다.

</details>

In [ ]:
# 8절 실험을 정리한다. 노드를 지우고 이 절이 만든 제약 두 개를 이름으로 내린다
run_cypher("MATCH (n) WHERE n:ExNameKey OR n:ExIdKey OR n:ExUniqueDemo OR n:ExNodeKeyDemo "
           "DETACH DELETE n")
for _name in ['ex_unique_demo', 'ex_nodekey_demo']:
    run_cypher("DROP CONSTRAINT " + _name + " IF EXISTS")
print('남은 8절 노드:', run_cypher("MATCH (n) WHERE n:ExNameKey OR n:ExIdKey OR n:ExUniqueDemo "
                                "OR n:ExNodeKeyDemo RETURN count(n) AS n")[0]['n'], '개')

---
# 9. 설계의 함정: 한 노드에 몰릴 때

4절에서 연도를 노드로 올렸습니다. 좋은 결정이었습니다. 그런데 그 결정에는 값이 붙습니다. 1894년처럼 **일이 많이 일어난 해**의 노드에는 사건이 계속 붙습니다. 자료를 넓힐수록 더 붙습니다. 이렇게 관계가 유난히 많이 붙은 노드를 **밀집 노드(supernode)** 라고 부릅니다.

<img src="images/supernode.png" width="880">

무엇이 문제인지 숫자로 봅시다.

> 아래 셀이 만드는 `ExDenseEvent` 노드 500개는 **모양만 보려고 만든 더미**입니다. 실제 사건이 아니라 번호만 붙였습니다. 이 교안에서 지어낸 값은 여기뿐입니다.

In [ ]:
# 사건이 몰린 해와 그렇지 않은 해를 나란히 만든다. 여기까지는 실제 사건이다
run_cypher("""
MERGE (y1:ExYear {value: 1776})
MERGE (y2:ExYear {value: 1894})
MERGE (a:ExEvent {name: '정조 즉위'})      MERGE (a)-[:HAPPENED_IN]->(y1)
MERGE (b:ExEvent {name: '미국 독립선언'})  MERGE (b)-[:HAPPENED_IN]->(y1)
MERGE (c:ExEvent {name: '청일전쟁 발발'})  MERGE (c)-[:HAPPENED_IN]->(y2)
MERGE (d:ExEvent {name: '갑오개혁'})       MERGE (d)-[:HAPPENED_IN]->(y2)
MERGE (e:ExEvent {name: '동학농민운동'})   MERGE (e)-[:HAPPENED_IN]->(y2)
""")
# 여기부터가 더미다. 1894 쪽에만 번호만 붙인 노드 500개를 매단다
run_cypher("""
MATCH (y:ExYear {value: 1894})
UNWIND range(1, 500) AS i
MERGE (e:ExDenseEvent {no: i})
MERGE (e)-[:HAPPENED_IN]->(y)
""")
# 연도마다 몇 개가 붙었는지 센다. 이 개수를 그 노드의 '차수' 라고 부른다
for row in run_cypher("MATCH (y:ExYear)<-[:HAPPENED_IN]-(x) "
                      "RETURN y.value AS 해, count(x) AS 붙은사건 ORDER BY 해"):
    print(row)

In [ ]:
# 그 노드를 가운데 두고 '같은 해에 있었던 두 사건' 을 세어 본다
# a <> b 는 자기 자신과 짝짓지 않게 하는 조건이다. 순서를 주지 않았으니 같은 두 사건이
# 앞뒤를 바꿔 두 번 세어진다
for row in run_cypher("MATCH (a)-[:HAPPENED_IN]->(y:ExYear)<-[:HAPPENED_IN]-(b) "
                      "WHERE a <> b "
                      "RETURN y.value AS 해, count(*) AS 짝 ORDER BY 해"):
    print(row)

> 붙은 사건은 2건과 503건으로 **250배쯤** 차이인데, 짝은 2건과 252,506건으로 **12만 배** 넘게 벌어졌습니다. 밀집 노드를 가운데 두는 공유 패턴은 붙은 개수의 **제곱**으로 자랍니다. 5,000개가 붙으면 짝이 2,500만 개입니다. 쿼리 하나가 데이터베이스를 몇 분씩 붙잡습니다.

> 그 노드를 **거쳐 가기만** 할 때도 값이 붙습니다. 노드에 닿은 뒤 반대쪽으로 나가려면 붙은 관계를 다 훑어야 합니다. "1894년에 무슨 일이 있었나"는 503건을 읽으면 끝나지만, "1894년을 거쳐 다른 무언가로"는 503건마다 그다음을 봅니다.

## 어떻게 피하나요

| 이럴 때 | 어떻게 |
|---|---|
| 그 노드를 **거쳐 갈 일이 없다** | 노드로 올리지 말고 속성으로 둔다 |
| 묶는 단위가 **너무 굵다** | 더 잘게 쪼갠다(해 대신 해 → 달 → 날) |
| 종류가 섞여 몰린다 | **관계 타입을 나눠** 훑는 범위를 좁힌다 |
| 한쪽 방향만 쓴다 | 화살표를 붙여 매칭해 반대쪽을 훑지 않게 한다 |
| 그래도 몰린다 | 그 노드를 **가운데 두는 공유 패턴**을 피한다. 한쪽에서 출발해 필요한 만큼만 넓힌다 |
| 질의를 짤 때 | 그 노드를 **경로의 끝**에 두고 **시작점으로 삼지 않는다** |

> 밀집 노드는 **설계가 틀렸다는 신호가 아닙니다.** 연도를 노드로 올린 것은 여전히 좋은 결정입니다. 다만 그 노드를 **가운데 두고 양쪽으로 뻗는 질의**를 조심하라는 뜻입니다. 설계할 때 "여기에 몇 개나 붙을 것 같은가"를 한 번 세어 보는 습관이 필요합니다.

**그럼 의료 데이터에서는?** 의료 그래프에서 "승인된 약"을 `:Approved` 노드 하나로 만들어 모든 약을 거기에 이어 두면 그 노드가 곧바로 밀집 노드가 됩니다. 그래서 승인 여부 같은 구분은 노드가 아니라 **속성이나 라벨**로 둡니다(6절). 값이 적고 잘 안 바뀌는 구분을 노드로 올리면 얻는 것 없이 밀집 노드만 생깁니다.

### ✅ 바로 확인 퀴즈

**1.** 밀집 노드에서 "같은 것을 가리키는 두 개체"를 찾는 질의가 왜 특히 비싼가요?

<details><summary>정답 보기</summary>

짝의 수가 붙은 관계 수의 **제곱**으로 자라기 때문입니다. 500개가 붙으면 짝이 25만, 5,000개가 붙으면 2,500만입니다. 노드 하나에 붙은 수가 열 배 늘면 결과는 백 배 늘어납니다.

</details>

**2.** 연도 노드가 밀집 노드가 될 것 같다면 어떤 선택지가 있나요? 두 가지를 드세요.

<details><summary>정답 보기</summary>

1. **더 잘게 쪼갭니다.** 해 하나에 몰리는 대신 해 → 달 → 날로 층을 두면 한 노드에 붙는 수가 줄어듭니다.
2. **거쳐 갈 일이 없다면 속성으로 되돌립니다.** 연도로 묶어 세기만 한다면 `e.year` 속성으로도 됩니다. 노드로 올리는 값은 그 노드를 **징검다리로 쓸 때** 나옵니다.

</details>

In [ ]:
# 9절 실험 노드를 지운다. 더미 500개도 여기서 사라진다
run_cypher("MATCH (n) WHERE n:ExYear OR n:ExEvent OR n:ExDenseEvent DETACH DELETE n")
print('남은 9절 노드:', run_cypher("MATCH (n) WHERE n:ExYear OR n:ExEvent OR n:ExDenseEvent "
                                "RETURN count(n) AS n")[0]['n'], '개')

---
## 📋 설계 검토 체크리스트

설계를 끝냈다고 생각될 때 이 목록을 훑으세요. 적재하고 나면 고치는 값이 비싸집니다.

**질문**
- [ ] 답하고 싶은 질문을 **문장으로** 적었는가
- [ ] 그중 **2홉 이상**인 질문이 있는가(하나도 없으면 관계형이 낫다)
- [ ] 깊이를 **모르는** 가변 길이 질문이 있는가

**노드와 라벨**
- [ ] 그 값을 **거쳐 건너갈** 일이 있는가(있으면 노드)
- [ ] 그 값 **자체에 붙일 것**이 있는가(있으면 노드)
- [ ] 라벨은 "이 노드가 무엇인가"만 담는가(값을 라벨로 두지 않았는가)
- [ ] 노드 하나에 라벨이 셋을 넘지 않는가

**관계**
- [ ] 뜻이 다른 연결을 한 타입에 몰아넣지 않았는가
- [ ] 타입 이름이 동사구이고, 방향과 함께 읽으면 문장이 되는가(`:HAS`·`:RELATED_TO` 가 없는가)
- [ ] 대칭 관계를 양쪽으로 두 번 저장하지 않았는가
- [ ] 셋 이상이 얽힌 사실을 **중간 노드**로 올렸는가

**속성과 키**
- [ ] 속성 이름이 그 값이 무엇인지 말하는가(`date` 대신 `signed_on`)
- [ ] **아는 정밀도까지만** 저장했는가
- [ ] **무엇이 같으면 같은 것인지** 정했는가
- [ ] 그 키가 **어느 범위 안에서** 유일한지 적었는가
- [ ] 키를 **제약으로** 잠갔는가
- [ ] 한 노드에 관계가 **몰리는 자리**가 없는가

In [ ]:
# 이 노트북이 만든 실험 노드가 하나도 남지 않았는지 마지막으로 확인한다
# 절마다 지웠으니 0 이 나와야 한다. 0 이 아니면 어느 절의 정리 셀을 건너뛴 것이다
EXPERIMENT_LABELS = ("n:ExKing OR n:ExKinng OR n:ExPerson OR n:ExReign OR n:ExThrone "
                     "OR n:ExEvent OR n:ExScopeEvent OR n:ExYear OR n:ExDenseEvent "
                     "OR n:FlatKing OR n:NodeKing OR n:ExNameKey OR n:ExIdKey "
                     "OR n:ExUniqueDemo OR n:ExNodeKeyDemo")
print('남은 실험 노드:', run_cypher("MATCH (n) WHERE " + EXPERIMENT_LABELS +
                                 " RETURN count(n) AS n")[0]['n'], '개')

## 이번 강의 정리

| 결정 | 기준 | 오늘 고른 답 |
|---|---|---|
| 밑그림 | 질문의 **명사에서 노드**, **동사에서 관계** | 답이 안 나오는 질문이 다듬을 자리를 알려 준다 |
| 스키마 | 문서로 정하고, 지킬 수 있는 것만 제약으로 잠근다 | 강제되는 것은 **키 제약뿐** |
| 무엇부터 정하나 | **질문**이 모양을 정한다 | 2홉 이상이 없으면 관계형 |
| 노드냐 속성이냐 | 거쳐 건너갈 일이 있는가, 그 값 자체에 붙일 것이 있는가 | 재위는 **노드** |
| 날짜 | **무엇의** 날짜인지 이름에 적고, **아는 정밀도까지만** | 조인일과 발효일을 따로 |
| 관계 타입 | 뜻이 다르면 나눈다 | **생부와 양자**를 따로 |
| 관계 방향 | 뜻이 자연스러운 쪽으로 한 번만 | 대칭이면 한 방향만 저장 |
| 라벨이냐 속성이냐 | 가짓수가 적고 잘 안 바뀌면 라벨 | 종류는 라벨, 바뀌는 값은 속성 |
| 중간 노드 | 셋 이상이 얽히면 올린다 | 재위를 노드로 올려 사건을 이었다 |
| 키 | **무엇이 같으면 같은 것인가** | 이름 말고 대수(범위는 조선 안) |
| 제약 | 키가 비는 것까지 막으려면 `NODE KEY` | `UNIQUE` 는 빈 키를 통과시킨다 |
| 밀집 노드 | 붙는 개수를 미리 세어 본다 | 공유 패턴은 **제곱**으로 자란다 |

- **두 안을 다 적재해 같은 질문을 던져 보는 것**이 이 교안의 방법이었습니다. 표만 보고 고르지 않았습니다.
- 값을 **거쳐 건너갈** 일이 있으면 노드입니다. 속성은 그 값에서 다른 데로 갈 수 없습니다.
- **뜻이 다르면 타입을 나눕니다.** 속성으로 구분하면 조건을 빠뜨린 쿼리가 조용히 틀린 답을 냅니다.
- **정답이 하나인 문제가 아닙니다.** 무엇을 물을 것인가가 정합니다. 그래서 설계의 첫 단계가 질문 적기입니다.

## ⏭️ 다음 교안

모양을 정했으니 이제 **넣습니다.** 다음 교안은 파일을 그래프로 옮기는 두 가지 길을 다룹니다. `LOAD CSV` 로 **서버가 파일을 읽는** 길과, 파이썬 **드라이버로 보내는** 길입니다. 노드 15,540개와 관계 91,966개가 실제로 들어가고, 도메인은 **의료 지식그래프**로 바뀝니다. 오늘 절마다 놓아 둔 "그럼 의료 데이터에서는?" 문단이 그 자료의 설계를 미리 설명한 것입니다.

적재하고 나면 오늘 정한 것들이 **적재 속도와 검증 방법으로** 돌아옵니다. 관계 타입을 나눠 둔 덕에 적재 쿼리에서 양 끝의 라벨을 지정할 수 있고, 그것이 적재 시간을 크게 가릅니다.

수고하셨습니다!